# Модель классификации для задачи оценки риска


## Общая схема и оглавление

Разобьём решение задачи на несколько этапов

1. **Постановка задачи** — уточняем, какие признаки у нас есть, что означают их значения и что будем предсказывать
2. **Сбор данных** — сбор датасета: в нашем случае пропускаем, ведь датасет дали как часть задания
3. **Анализ данных и выбор спецификации модели** — проводим EDA и определяем устройство будущей модели
   - Базовый **EDA**: знакомимся с данными, проверяем ошибки и смотрим на каждый признак отдельно
   - Делим данные на **train**, **validation**, **test**
   - Попарный анализ на train — исследуем все пары переменных, включая целевую
   - Проверка гипотез на train — проверяем предположения из предыдущих выводов, в том числе зависимости трёх и более переменных
   - Выбираем признаки, способы их обработки и варианты спецификации моделей, которые сравним после обучения
4. **Финальная предобработка** — подготавливаем данные с учётом спецификации модели и выводов EDA
5. **Обучение** — обучаем модели и выбираем гиперпараметры по validation
6. **Оценка** — проверяем выбранное решение на test
7. **Эксплуатация** — проверяем на реальных данных

## Подготовка зависимостей

Требуется Python `>=3.13`. Следующая ячейка устанавливает библиотеки без закрепления версий и без принудительного обновления

**Поместите файл `credit_risk_dataset.csv` в ту же папку, что и ноутбук** и запускайте ноутбук из этой папки. В Google Colab загрузите CSV в рабочую папку среды выполнения через панель файлов

In [ ]:
%pip install pandas numpy scipy scikit-learn threadpoolctl matplotlib seaborn

from pathlib import Path

dataset_path = Path("credit_risk_dataset.csv")

## Вспомогательные функции

Тут будут всякие функции-помощники, чтобы не отвлекаться на них в блоках кода дальше

In [ ]:
# 12.340 -> 12.34; 18.000 -> 18
def fmt_number(x):
    return f'{x:.2f}'.rstrip('0').rstrip('.')


# B, A, B -> {A, B}
def print_values_set(values):
    return '{' + ', '.join(str(v) for v in sorted(values.unique())) + '}'


# 18.00, 100.00 -> [18, 100]
def print_interval(lower, upper):
    return f'[{fmt_number(lower)}, {fmt_number(upper)}]'

# 12345 -> 12 345
def fmt_thousands(value):
    return f'{value:,.0f}'.replace(',', ' ')


# 12.345 -> 12.35 %; значения ниже min_share -> пустая строка
def fmt_percent(share, min_share=0):
    return f'{share:.2f} %' if share >= min_share else ''


# 1234, 12.345 -> 1 234 (12.35 %)
def fmt_count_share(count, share):
    return f'{fmt_thousands(count)} ({fmt_percent(share)})'


## Подготовка формата

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, PercentFormatter

# Убираем хвостовые нули при выводе чисел: 12.340000 -> 12.34
pd.set_option('display.float_format',
              lambda x: f'{x:.6f}'.rstrip('0').rstrip('.'))

# Показываем текст в ячейках таблиц целиком, без обрезки
pd.set_option('display.max_colwidth', None)

# Размещение графиков распределений с учётом подписей
PLOT_LAYOUT = {'constrained_layout': True}

# Общие цвета распределений, boxplot, медианы и категорий
DISTRIBUTION_COLOR = '#3977a8'
BOX_COLOR = '#91bad7'
MEDIAN_COLOR = '#b34c32'
CATEGORY_COLORS = ['#3977a8', '#e6a34a', '#66a68b', '#b578a5']

# Подписи денежной шкалы: 12345 -> 12 345
money_format = FuncFormatter(lambda value, _: fmt_thousands(value))


## 1. Постановка задачи

**Входы**

- Задание, датасет и описания полей с Kaggle

**Выходы**

- Что предсказываем и какие данные подаём модели
- Кому нужен прогноз и когда его можно строить
- Требования к данным
- Критерии выбора модели и порога, метрики итогового качества

**Что делаем**

Загрузим данные и разберёмся, что в них записано. После этого уточним задачу и решим, как оценивать результат\
Этот этап по сути включает начало EDA – но только потому, что в качестве формулировки мы получили не предметную задачу, а сами данные, из которых эту задачу формулируем. В обычных условиях было бы наоборот

### Исходное задание

```text
задание

1. Скачать набор экономических данных по ссылке
https://www.kaggle.com/datasets/laotse/credit-risk-dataset.
2. Загрузить данные с использованием Pandas или специализированных библиотек.
3. Провести Разведочный анализ данных и визуализацию.
4. Провести предобработку данных.
5. Реализовать модель классификации для задачи оценки риска
6. Оценить качество модели с использованием соответствующих метрик.
7. Загрузить .ipynb - файл с программой
```

Источник описаний полей: https://www.kaggle.com/datasets/laotse/credit-risk-dataset


### Исходные данные и описания полей

Данные загрузим из CSV, взятого с Kaggle, оттуда же приведу исходные описания столбцов

In [ ]:
data_raw = pd.read_csv(dataset_path)
data = data_raw.copy()


<table>
<thead><tr><th>Столбец</th><th>Описание автора</th></tr></thead>
<tbody>
<tr><td><code>person_age</code></td><td>Age</td></tr>
<tr><td><code>person_income</code></td><td>Annual Income</td></tr>
<tr><td><code>person_home_ownership</code></td><td>Home ownership</td></tr>
<tr><td><code>person_emp_length</code></td><td>Employment length (in years)</td></tr>
<tr><td><code>loan_intent</code></td><td>Loan intent</td></tr>
<tr><td><code>loan_grade</code></td><td>Loan grade</td></tr>
<tr><td><code>loan_amnt</code></td><td>Loan amount</td></tr>
<tr><td><code>loan_int_rate</code></td><td>Interest rate</td></tr>
<tr><td><code>loan_status</code></td><td>Loan status (0 is non default 1 is default)</td></tr>
<tr><td><code>loan_percent_income</code></td><td>Percent income</td></tr>
<tr><td><code>cb_person_default_on_file</code></td><td>Historical default</td></tr>
<tr><td><code>cb_person_cred_hist_length</code></td><td>Credit history length</td></tr>
</tbody>
</table>

### 1.1 Предварительный анализ

**Входы**

- Загруженные данные и короткие описания столбцов

**Выходы**

- Понятные описания полей и их значений
- Типы хранения и значения столбцов

**Что делаем**

Смотрим, что записано в каждом столбце и как читать его значения

##### Что можно понять из описаний полей

У автора датасета описания короткие: не везде указаны единицы измерения и расшифровки категорий. Посмотрим, какие значения есть в таблице, и уточним, что означает каждый столбец

##### Типы и значения столбцов

Выведем все столбцы одной таблицей: тип хранения и диапазон либо встречающиеся значения

In [ ]:
def domain(column):
    values = data[column].dropna()
    if pd.api.types.is_numeric_dtype(values):
        return print_interval(values.min(), values.max())
    return print_values_set(values)


columns_info = pd.DataFrame(index=data.columns)
columns_info['Тип в данных'] = data.dtypes.astype(str)
columns_info['Домен'] = columns_info.index.map(domain)
display(columns_info)


##### Что означают столбцы

Сопоставим описания автора с тем, что увидели в данных

**Дефолт** — ситуация, когда заёмщик не выполняет обязательства по кредиту: перестаёт платить или допускает существенную просрочку

<table>
<thead><tr><th>Столбец</th><th>Как понимаем значения</th><th>Тип</th></tr></thead>
<tbody>
<tr><td><code>person_age</code></td><td>возраст заёмщика, лет</td><td>Числовой</td></tr>
<tr><td><code>person_income</code></td><td>годовой доход</td><td>Числовой</td></tr>
<tr><td><code>person_home_ownership</code></td><td>жильё:<br>• RENT — снимает<br>• OWN — своё<br>• MORTGAGE — в ипотеке<br>• OTHER — другое</td><td>Номинальный</td></tr>
<tr><td><code>person_emp_length</code></td><td>стаж на текущей работе, лет</td><td>Числовой</td></tr>
<tr><td><code>loan_intent</code></td><td>цель кредита:<br>• PERSONAL — личные нужды<br>• EDUCATION — образование<br>• MEDICAL — лечение<br>• VENTURE — бизнес<br>• HOMEIMPROVEMENT — ремонт жилья<br>• DEBTCONSOLIDATION — рефинансирование других долгов</td><td>Номинальный</td></tr>
<tr><td><code>loan_grade</code></td><td>грейд займа от A (надёжный) до G (рискованный)</td><td>Порядковый</td></tr>
<tr><td><code>loan_amnt</code></td><td>сумма кредита</td><td>Числовой</td></tr>
<tr><td><code>loan_int_rate</code></td><td>процентная ставка, % годовых</td><td>Числовой</td></tr>
<tr><td><code>loan_status</code></td><td>статус кредита:<br>• 0 — платит<br>• 1 — дефолт</td><td>Бинарный</td></tr>
<tr><td><code>loan_percent_income</code></td><td>доля кредита от годового дохода</td><td>Числовой</td></tr>
<tr><td><code>cb_person_default_on_file</code></td><td>были ли дефолты раньше (по данным кредитного бюро):<br>• Y — были<br>• N — не было</td><td>Бинарный</td></tr>
<tr><td><code>cb_person_cred_hist_length</code></td><td>длина кредитной истории, лет</td><td>Числовой</td></tr>
</tbody>
</table>

### 1.2 Формулировка

**Входы**

- Задание и результаты первого знакомства с данными

**Выходы**

- Что предсказываем и по каким данным
- Требования к данным
- Полный набор метрик качества и правила выбора модели и порога

**Что делаем**

Собираем постановку задачи: что подаём модели, что хотим получить и как поймём, насколько хорошо она работает

#### Для кого и когда строим прогноз

Предположим, что заёмщик уже подал заявку на кредит. Кредитор оценил её, назначил грейд `loan_grade` и предложил ставку `loan_int_rate`\
Модель нужна инвестору, который решает, финансировать ли кредит с уже назначенными грейдом и ставкой. В этом сценарии оба признака известны до прогноза, поэтому оставляем их. Сам сценарий — наше допущение: автор не уточняет, как и на каких данных кредитор назначал грейд и ставку

Целевая переменная — `loan_status`: 0 — без дефолта, 1 — дефолт. Будем предсказывать её по остальным столбцам\
Решаем задачу бинарной классификации. Модель выдаёт оценку риска, которую по выбранному порогу переводим в класс 0 или 1

#### Что подаём модели и что получаем

| Что определяем | В нашем случае |
| --- | --- |
| Одна запись | Заявка на кредит |
| Вход | Все 11 признаков, кроме `loan_status` |
| Результат | Оценка риска, а после применения порога — класс 0 или 1 |

Алгоритм пока не выбираем: сначала разберём данные

#### Какие данные нужны

Для обучения нужен известный исход кредита — `loan_status`, это целевая переменная\
Для новых заявок нужны те же 11 полей в исходных единицах измерения

Используем только то, что известно в момент прогноза. В нашем сценарии грейд и ставка уже есть, а фактического исхода кредита ещё нет\
Пропуски, повторы и подозрительные значения проверим при EDA. Период, за который автор определял дефолт, не указан: модель будет предсказывать метку в том виде, в котором она дана в источнике

#### Критерии оценивания

Положительный класс — дефолт: `loan_status = 1`. По AP выберем модель, по F2 — порог. Ниже зададим метрики, которые рассчитаем при итоговой оценке

| Обозначение | Число кредитов |
| --- | --- |
| **TP** | Верно предсказан дефолт |
| **FP** | Предсказан дефолт, фактически его нет |
| **FN** | Дефолт не предсказан, фактически он произошёл |
| **TN** | Верно предсказано отсутствие дефолта |

##### Метрики по предсказанным классам

| Метрика | Формула | Что показывает |
| --- | --- | --- |
| Accuracy | $\frac{TP+TN}{TP+FP+FN+TN}$ | Доля верных ответов |
| Precision (точность) | $\frac{TP}{TP+FP}$ | Доля настоящих дефолтов среди предсказанных |
| Recall (полнота) | $\frac{TP}{TP+FN}$ | Доля обнаруженных дефолтов среди всех фактических |
| F1 | $\frac{2TP}{2TP+FP+FN}$ | Объединяет precision и recall с равными весами |
| F2 | $\frac{5TP}{5TP+FP+4FN}$ | Объединяет precision и recall с большим весом recall |
| Balanced accuracy | $\frac{1}{2}\left(\frac{TP}{TP+FN}+\frac{TN}{TN+FP}\right)$ | Средняя полнота по двум классам |

Для всех метрик этой группы больше — лучше

##### Метрики по оценкам риска

**Average Precision (AP)** — среднее precision, взвешенное приростами recall при снижении порога

$$AP=\sum_i (R_i-R_{i-1})P_i,\qquad P_i=\frac{TP_i}{TP_i+FP_i},\quad R_i=\frac{TP_i}{TP_i+FN_i}$$

Здесь $i$ — номер порога при последовательном снижении, $P_i$ и $R_i$ — precision и recall при этом пороге, $R_0=0$. AP принимает значения от 0 до 1; больше — лучше

**ROC-AUC** — площадь под ROC-кривой. При разных порогах она сопоставляет полноту (TPR) с долей ложных тревог среди кредитов без дефолта (FPR):

$$TPR=\frac{TP}{TP+FN},\qquad FPR=\frac{FP}{FP+TN}$$

Для бинарной классификации ROC-AUC также можно рассчитать по всем парам «дефолт — без дефолта»:

$$\mathrm{ROC\!\text{-}\!AUC}=\frac{N_{>}+0{,}5N_{=}}{n_+n_-}$$

Здесь $n_+$ — число кредитов с дефолтом, $n_-$ — без дефолта; $N_{>}$ — число пар, в которых оценка риска дефолтного кредита выше, $N_{=}$ — число пар с равными оценками. Правильно упорядоченная пара даёт 1, равные оценки — 0,5, обратный порядок — 0\
Значения от 0 до 1; больше — лучше. ROC-AUC = 1 означает, что все дефолты получили оценки выше всех кредитов без дефолта; 0,5 соответствует случайному ранжированию. Как и AP, эта метрика не привязана к одному выбранному порогу

**Brier score** — средняя квадратичная ошибка прогноза вероятности

$$\mathrm{Brier}=\frac{1}{n}\sum_{j=1}^{n}(s_j-y_j)^2$$

Здесь $s_j$ — прогноз вероятности дефолта, $y_j\in\{0,1\}$ — фактический исход, $n$ — число кредитов. Значения от 0 до 1; меньше — лучше

##### Правила выбора и итоговой оценки

Основной критерий сравнения моделей — AP\
Пропустить будущий дефолт хуже, чем ошибочно отнести надёжную заявку к рискованным. Поэтому при выборе порога отдаём приоритет полноте и используем F2, которая сильнее учитывает recall

| Что оцениваем или выбираем | Правило |
| --- | --- |
| Модель | Наибольшая AP на validation |
| Порог | Наибольшая F2 на validation для выбранной модели; при равенстве — больший порог |
| Итоговое качество | На test рассчитываем AP, ROC-AUC, Brier, Accuracy, Precision, Recall, F1, F2 и Balanced accuracy, показываем TP, FP, FN и TN |

## 2. Сбор данных

**Входы**

- Требования к данным
- Доступные источники

**Выходы**

- Датасет для анализа и обучения модели

**Что делаем**

На этом этапе нужно по требованиям к данным, выясненным на прошлом этапе, собирать датасет из источников. Но в нашем случае датасет – часть формулировки задачи, поэтому пропускаем этап (:

## 3. Анализ данных и выбор спецификации модели

**Входы**

- Данные и постановка задачи

**Выходы**

- Спецификация будущей модели и варианты, которые будем сравнивать при обучении
- Таблица выводов с накопленными знаниями о данных и принятыми решениями
- Подготовленные для дальнейшей работы данные и выборки train, validation и test

**Что делаем**

Основная задача — провести EDA, разобраться в данных и на этой основе выбрать спецификацию будущей модели\
**Спецификация модели** — состав признаков, способы их обработки, семейство моделей и рассматриваемые настройки\
Сейчас выберем признаки, обработку и варианты моделей. Обучим их на этапе 5

### Как исследуем зависимости

После базового EDA и разбиения исследование зависимостей делим на два подэтапа:

- **Попарный анализ** — проверяем связи всех пар переменных, включая пары с `loan_status`
- **Проверка гипотез** — проверяем предположения из предыдущих выводов о совместных зависимостях трёх и более переменных, а также вопросы о форме связей, на которые одного коэффициента недостаточно

Перед расчётами соберём гипотезы в таблицу. В неё включим и некоторые предположения о двух переменных. Таблица фиксирует вопросы из прошлых наблюдений, но не ограничивает попарный анализ: в нём проверим все пары, даже если отдельной гипотезы для них нет

### Подэтапы

- **3.1 Базовый EDA** — общий обзор, проверка данных и анализ отдельных признаков
- **3.2 Разбиение данных** — выделение train, validation и test
- **3.3 Корреляционный анализ на train** — все попарные связи, включая целевую переменную
- **3.4 Проверка гипотез на train** — предположения из предыдущих выводов о совместных зависимостях и форме связей
- **3.5 Выбор спецификации модели** — состав признаков, обработка и варианты моделей для последующего сравнения

### Вспомогательные шаги

**Делим данные перед анализом корреляции признаков**, анализ проводим только на train, чтобы test и validation не утекали в решения, принятые на основе этого анализа

**Часть обработки делаем по ходу анализа**: удаляем повторы, исправляем явные ошибки и дальше работаем с очищенными данными. После преобразования снова смотрим на результат\
Финальную предобработку под выбранную спецификацию выполняем на этапе 4

### Таблица выводов

Наблюдения и решения собираем в общую таблицу `conclusions`. После каждого подэтапа дополняем её или уточняем прежние выводы, чтобы к обучению иметь актуальный набор решений\
Новые строки выделяем зелёным и знаком «+», заменённые один раз показываем красным со знаком «−». В конце оставляем только актуальные выводы

In [ ]:
# Одна таблица выводов для последовательных шагов анализа
import pandas as pd


def empty_findings():
    return pd.DataFrame(columns=['Выводы', 'Решения', 'Метка'], index=pd.Index([], name='id'), dtype=object)


def update_findings(table, additions=(), remove=()):
    # Записи: (id, вывод, решение) Метки new/removed действуют до следующего обновления
    result = table.loc[table['Метка'].ne('removed')].copy()
    result['Метка'] = ''
    for key in remove:
        if key in result.index:
            result.loc[key, 'Метка'] = 'removed'
    for key, finding, decision in additions:
        if key in result.index:
            old = result.loc[key]
            if old['Выводы'] != finding or old['Решения'] != decision:
                previous_key = f'{key}::previous'
                result = result.rename(index={key: previous_key})
                result.loc[previous_key, 'Метка'] = 'removed'
        result.loc[key] = [finding, decision, 'new']
    return result


def active_findings(table):
    # Актуальные строки без служебных меток и удаляемых версий
    return table.loc[table['Метка'].ne('removed'), ['Выводы', 'Решения']].copy()


def findings_view(table, only_new=False):
    # Полная таблица или только новые строки с прежними цветами и знаками
    selected = table.loc[table['Метка'].eq('new')] if only_new else table
    view = selected[['Выводы', 'Решения']].copy()
    signs = selected['Метка'].map({'new': '+ ', 'removed': '− ', '': ''}).fillna('')
    view['Выводы'] = signs + view['Выводы']
    colors = selected['Метка'].map({
        'new': 'background-color: #dcefe3; color: #172b20',
        'removed': 'background-color: #f9dddd; color: #7a2020',
        '': '',
    }).fillna('')
    return (view.style
            .apply(lambda frame: pd.DataFrame({column: colors for column in frame.columns}, index=frame.index), axis=None)
            .format(escape='html')
            .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap', 'vertical-align': 'top'})
            .hide(axis='index'))


In [ ]:
conclusions = empty_findings()

### 3.1 Базовый EDA

**Входы**

- Исходная таблица

**Выходы**

- Очищенные данные
- Первые наблюдения и решения

**Что делаем**

Сначала посмотрим на таблицу в целом, затем проверим ошибки и разберём каждый столбец отдельно

#### 3.1.1 Общий обзор

**Входы**

- Таблица и описания столбцов

**Выходы**

- Общая характеристика данных и первые наблюдения

**Что делаем**

Посмотрим, сколько у нас данных, какие поля есть и где встречаются пропуски

##### Общий взгляд на данные

##### Посмотрим размер таблицы и несколько записей из разных её частей

In [ ]:
data_shape = data.shape
base_props = pd.DataFrame([data_shape[0], data_shape[1], data_shape[1] - 1, 1], index=['Записей',
                         'Переменных', 'Признаков', 'Целевых переменных'], columns=['Количество'])

display(base_props)
display(data.head(3))
display(data.sample(3))
display(data.tail(3))


Для каждого столбца посмотрим пропуски, тип и встречающиеся значения

In [ ]:
columns_table = columns_info[['Домен', 'Тип в данных']].copy()
columns_table.insert(0, 'Пропусков', data.isna().sum())
display(columns_table)

#### Выводы после общего обзора

In [ ]:
conclusions = update_findings(conclusions, [
    ('f01', 'В датасете 32 581 запись, 11 признаков и одна целевая переменная', '—'),
    ('f02', 'Среди признаков: 7 числовых, 2 номинальных, 1 порядковый и 1 бинарный', '—'),
    ('f03', 'В person_emp_length (2,75 % записей) и в loan_int_rate (9.56% записей) есть пропуски', '—'),
    ('f04', 'Целевая переменная заполнена во всех записях и содержит только значения 0 и 1', '—'),
    ('f05', 'Возраст 144 года и стаж 123 года выглядят как ошибки в данных', '—'),
    ('f06', 'Числовые признаки имеют существенно разные масштабы: максимум дохода — 6 000 000, максимум доли кредита от дохода — 0,83', '—'),
])
display(findings_view(conclusions))

#### 3.1.2 Проверка и очистка

**Входы**

- Исходные данные

**Выходы**

- Очищенные данные
- Решения о том, что убрать или исправить

**Что делаем**

Проверим одинаковые строки, подозрительные значения и согласованность связанных полей

##### Проверим данные

Идентификаторов заявок нет, поэтому одинаковые значения могут принадлежать разным заявкам. Проверим число повторов в каждой группе и расположение копий в файле\
Отдельно проверим, встречаются ли одинаковые признаки с разным `loan_status`: это уже не полное совпадение записей

In [ ]:
duplicate_extra = data.duplicated()
duplicate_all = data.duplicated(keep=False)
feature_columns = data.columns.drop('loan_status')
conflicting_groups = (
    data.groupby(list(feature_columns), dropna=False)['loan_status']
        .nunique().gt(1)
)

duplicate_summary = pd.DataFrame({
    'Количество': [
        int(duplicate_extra.sum()),
        int(duplicate_all.sum()),
        len(data.loc[duplicate_all].drop_duplicates()),
        int(conflicting_groups.sum()),
    ]
}, index=[
    'Повторные экземпляры сверх первого',
    'Все строки в повторяющихся группах',
    'Группы полностью одинаковых строк',
    'Группы одинаковых признаков с разными loan_status',
])
display(duplicate_summary)

repeated_groups = [
    positions for positions in data.groupby(list(data.columns), dropna=False).indices.values()
    if len(positions) > 1
]
display(pd.Series([len(positions) for positions in repeated_groups])
        .value_counts().sort_index().rename_axis('Записей в группе')
        .to_frame('Число групп'))
pair_gaps = pd.Series([
    positions[1] - positions[0] for positions in repeated_groups if len(positions) == 2
])
display(pd.Series({
    'Соседних пар': int(pair_gaps.eq(1).sum()),
    'Минимальное расстояние между копиями, строк': pair_gaps.min(),
    'Максимальное расстояние между копиями, строк': pair_gaps.max(),
}).to_frame('Значение'))
display(data.loc[duplicate_all].sort_values(list(data.columns)).head(10))


##### Проверим, согласуются ли значения между собой

Например, стаж не должен быть длиннее жизни человека, а доля кредита от дохода должна быть близка к отношению суммы кредита к доходу\
Ниже — рабочие границы проверки. Возраст 100 лет и начало работы в 14 лет — допущения для поиска подозрительных записей, а не правила автора датасета. Проверяем строки, в которых заполнены нужные поля

| Что проверяем | Условие | Почему |
| --- | --- | --- |
| Возраст | Не больше 100 лет | Более высокие значения требуют проверки |
| Стаж | Не больше 86 лет | 100 лет минус предполагаемый возраст начала работы 14 лет |
| Стаж и возраст | Стаж ≤ возраст − 14 | Предполагаем, что человек начал работать не раньше 14 лет |
| Кредитная история и возраст | История ≤ возраст − 16 | Предполагаем начало кредитной истории не раньше 16 лет |
| Доля кредита от дохода | Отличие от суммы / дохода не больше 0,005 | Учитываем округление доли до сотых |
| Возраст, стаж, кредитная история | Целые значения | Проверяем, записаны ли длительности в полных годах |

###### Условия проверки

Запишем перечисленные ограничения как условия нарушения

In [ ]:
ratio_expected = data['loan_amnt'] / data['person_income']
ratio_difference = (data['loan_percent_income'] - ratio_expected).abs()

checks = {
    'Целочисленность возраста': (
        ['person_age'], data['person_age'] % 1 != 0),
    'Целочисленность стажа': (
        ['person_emp_length'], data['person_emp_length'] % 1 != 0),
    'Целочисленность кредитной истории': (
        ['cb_person_cred_hist_length'], data['cb_person_cred_hist_length'] % 1 != 0),
    'Возраст ≤ 100': (
        ['person_age'], data['person_age'] > 100),
    'Стаж ≤ 86': (
        ['person_emp_length'], data['person_emp_length'] > 86),
    'Стаж ≤ возраст − 14': (
        ['person_age', 'person_emp_length'],
        data['person_emp_length'] > data['person_age'] - 14),
    'Кредитная история ≤ возраст − 16': (
        ['person_age', 'cb_person_cred_hist_length'],
        data['cb_person_cred_hist_length'] > data['person_age'] - 16),
    'Доля кредита соответствует сумме / доходу (погрешность ≤ 0,005)': (
        ['loan_percent_income', 'loan_amnt', 'person_income'],
        ratio_difference > 0.005 + 1e-9),
}



###### Число нарушений

Применяем условия только к строкам с нужными значениями и считаем число нарушений каждой проверки

In [ ]:
constraint_results = []
constraint_violations = pd.DataFrame(index=data.index)
for name, (columns, violation) in checks.items():
    available = data[columns].notna().all(axis=1)
    # Согласованность проверяем только при допустимом абсолютном стаже.
    if name == 'Стаж ≤ возраст − 14':
        available &= data['person_emp_length'] <= 86
    if 'person_income' in columns:
        available &= data['person_income'] > 0
    constraint_violations[name] = available & violation
    constraint_results.append({
        'Проверка': name,
        'Проверено': int(available.sum()),
        'Не проверено': int((~available).sum()),
        'Нарушений': int(constraint_violations[name].sum()),
    })

display(pd.DataFrame(constraint_results).set_index('Проверка').rename_axis(None))


##### Очистим найденные ошибки

Все 165 групп совпадающих строк — пары. Это позволяет предположить ошибочное дублирование при подготовке данных, поэтому оставляем по одной записи из каждой пары. Копии не стоят рядом в файле.

Также убираем записи с недопустимым возрастом. Недопустимый стаж заменяем пропуском; способ заполнения пропусков выберем позже\
Дальше работаем с очищенной таблицей `data`

In [ ]:
data = data_raw.drop_duplicates().copy()
invalid_age = data['person_age'] > 100
data = data.loc[~invalid_age].copy()

invalid_employment = data['person_emp_length'] > 86
data.loc[invalid_employment, 'person_emp_length'] = float('nan')


#### Выводы и решения после проверки корректности

In [ ]:
conclusions = update_findings(conclusions, [
    ('f03', 'В person_emp_length (2,75 % записей) и в loan_int_rate (9.56% записей) есть пропуски', 'Способ обработки пропусков выберем после анализа распределений'),
    ('f07', 'Найдено 165 групп ровно по две полностью одинаковые строки: 330 записей, из них 165 повторных (0,51 % выборки).\nВсе повторы парные; соседних копий нет. Считаем это признаком ошибочного дублирования при подготовке данных', 'Оставить по одной записи из каждой пары, исходя из предположения об ошибочном дублировании'),
    ('f08', 'Возраст превышает рабочую границу 100 лет в 5 записях: 144 года в трёх записях, 123 года в двух', 'Убрать эти 5 записей: их мало, а надёжно восстановить возраст нельзя'),
    ('f09', 'В двух записях стаж равен 123 годам и превышает выбранную границу 86 лет', 'Заменить недопустимый стаж пропусками, сохранив записи'),
    ('f10', 'Для записей со стажем ≤ 86 лет выполняется условие «стаж ≤ возраст − 14»', '—'),
    ('f11', 'Выполняется условие «кредитная история ≤ возраст − 16»', '—'),
    ('f12', 'В 429 записях (1,32 % всей выборки) доля кредита отличается от отношения суммы к доходу более чем на 0,005; одним округлением до сотых эти расхождения не объясняются', 'Проверим на этапе корреляционного анализа'),
    ('f13', 'Возраст, стаж и длина кредитной истории везде, где они указаны, записаны целыми числами', '—'),
    ('f26', 'После промежуточной очистки анализируем 32 411 записей', '—'),
], remove=['f05'])
display(findings_view(conclusions))

#### 3.1.3 Каждый признак отдельно

**Входы**

- Данные после очистки

**Выходы**

- Распределения признаков и наблюдения по ним

**Что делаем**

Смотрим, какие значения встречаются часто, какие редко и насколько велик разброс

##### Методология – как будем анализировать

Анализируем столбцы очищенной таблицы: для каждого посмотрим частые и редкие значения, пропуски и разброс. Для категорий выведем частоты и доли, для числовых признаков — описательную статистику, гистограмму и «ящик с усами». На выходе получим таблицы, графики и наблюдения по каждому признаку. Наблюдения и решения добавим в общую таблицу выводов

По гистограмме оценим форму распределения: высота столбца показывает число записей в интервале. Если длинный хвост мешает рассмотреть основную часть данных, дополнительно используем логарифмическую шкалу

По графику «ящик с усами» (**boxplot**) посмотрим медиану, разброс и статистические выбросы. Ящик охватывает среднюю половину наблюдений — от 25-го до 75-го процентиля, линия внутри показывает медиану. Разность границ ящика — **межквартильный размах**, или IQR:

$$IQR = Q_3-Q_1.$$

Значения за границами $[Q_1-1{,}5IQR;\;Q_3+1{,}5IQR]$ отметим как **статистические выбросы**. Усы тянутся до крайних наблюдаемых значений внутри этих границ, остальные значения показаны отдельными точками. Проверим, чем объясняются необычные значения, прежде чем решать, нужна ли обработка

По **асимметрии** оценим направление хвоста распределения: положительная — вправо, отрицательная — влево\
В таблицах посмотрим среднее (`mean`) и стандартное отклонение (`std`) — меру разброса вокруг среднего


##### 3.1.3.1 person_age


In [ ]:
age = data['person_age']
q1, q3 = age.quantile([0.25, 0.75])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
age_outliers = (age < lower_fence) | (age > upper_fence)

age_stats = pd.concat([
    age.describe(),
    pd.Series({
        'Коэффициент асимметрии': age.skew(),
        'Верхняя граница по правилу 1,5 IQR': upper_fence,
        'Записей за границами 1,5 IQR': int(age_outliers.sum()),
        'Доля записей за границами, %': 100 * age_outliers.mean(),
    }),
]).to_frame('Значение')
display(age_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), **PLOT_LAYOUT)
sns.histplot(x=age, discrete=True, color=DISTRIBUTION_COLOR, ax=axes[0])
axes[0].axvline(age.median(), color=MEDIAN_COLOR, linestyle='--',
               label=f'Медиана: {age.median():g}')
axes[0].set(title='Распределение возраста', xlabel='Возраст, лет', ylabel='Количество записей')
axes[0].legend()

sns.boxplot(x=age, color=BOX_COLOR, ax=axes[1])
axes[1].set(title='Возраст — ящик с усами', xlabel='Возраст, лет')
plt.show()


**Выводы и решения по person_age**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f14', 'Среди заёмщиков преобладают молодые, медиана возраста — 26 лет.\nБольшинство значений меньше; справа остаётся длинный хвост из более редких старших возрастов', '—'),
])
display(findings_view(conclusions, only_new=True))

##### 3.1.3.2 person_home_ownership


##### Тип жилья — person_home_ownership

Для номинального признака выводим частоты и доли категорий, затем строим столбчатую диаграмму

In [ ]:
ownership_counts = data['person_home_ownership'].value_counts()
ownership_table = pd.DataFrame({
    'Количество': ownership_counts,
    'Доля, %': 100 * ownership_counts / len(data),
})
display(ownership_table)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), **PLOT_LAYOUT)
colors = CATEGORY_COLORS
bars = axes[0].bar(ownership_table.index, ownership_table['Доля, %'], color=colors)
axes[0].bar_label(bars, labels=[
    fmt_count_share(count, share)
    for count, share in zip(ownership_table['Количество'], ownership_table['Доля, %'])
], padding=5, fontsize=9)
axes[0].set(title='Распределение по типу жилья', xlabel='Тип жилья', ylabel='Доля записей, %')
axes[0].set_ylim(0, ownership_table['Доля, %'].max() * 1.18)

wedges, _, _ = axes[1].pie(
    ownership_counts, colors=colors, startangle=90,
    autopct=lambda share: fmt_percent(share, min_share=1),
    pctdistance=0.72, wedgeprops={'edgecolor': 'white'},
)
axes[1].legend(wedges, [
    f'{category} — {fmt_percent(share)}'
    for category, share in ownership_table['Доля, %'].items()
], loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
axes[1].set_title('Доли категорий жилья')
plt.show()


**Выводы и решения по person_home_ownership**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f15', 'Категории типа жилья распределены неравномерно: RENT — 50,52 %, MORTGAGE — 41,24 %, OWN — 7,91 %, OTHER — 0,33 %', '—'),
])
display(findings_view(conclusions, only_new=True))

##### 3.1.3.3 person_income


##### Годовой доход — person_income

Сначала посмотрим статистику и распределение дохода в обычном масштабе

In [ ]:
import numpy as np

income = data['person_income']
income_q1, income_q3 = income.quantile([0.25, 0.75])
income_iqr = income_q3 - income_q1
income_lower = income_q1 - 1.5 * income_iqr
income_upper = income_q3 + 1.5 * income_iqr
income_outliers = (income < income_lower) | (income > income_upper)

income_stats = pd.concat([
    income.describe(),
    pd.Series({
        'Коэффициент асимметрии': income.skew(),
        'Верхняя граница по правилу 1,5 IQR': income_upper,
        'Записей за границами 1,5 IQR': int(income_outliers.sum()),
        'Доля записей за границами, %': 100 * income_outliers.mean(),
    }),
]).to_frame('Значение')
display(income_stats)

fig, income_linear_axes = plt.subplots(2, 1, figsize=(13, 7),
                                     gridspec_kw={'height_ratios': [3, 1]}, **PLOT_LAYOUT)
sns.histplot(x=income, bins=50, color=DISTRIBUTION_COLOR, ax=income_linear_axes[0])
income_linear_axes[0].set(title='Распределение годового дохода',
                          xlabel='Годовой доход', ylabel='Количество записей')
income_linear_axes[0].xaxis.set_major_formatter(money_format)
income_linear_axes[0].tick_params(axis='x', labelrotation=20)
sns.boxplot(x=income, color=BOX_COLOR, ax=income_linear_axes[1])
income_linear_axes[1].set(title='Доход — ящик с усами', xlabel='Годовой доход')
income_linear_axes[1].xaxis.set_major_formatter(money_format)
plt.show()


##### Вывод по обычной шкале

Из-за длинного правого хвоста основная часть записей сжата у левого края. Рассмотрим её на гистограмме с логарифмической шкалой дохода

In [ ]:
fig, income_log_ax = plt.subplots(figsize=(13, 4.5), **PLOT_LAYOUT)
log_bins = np.geomspace(income.min(), income.max(), 40)
sns.histplot(x=income, bins=log_bins, color=DISTRIBUTION_COLOR, ax=income_log_ax)
income_log_ax.set_xscale('log')
income_log_ax.axvline(income.median(), color=MEDIAN_COLOR, linestyle='--',
                      label=f'Медиана: {fmt_thousands(income.median())}')
income_log_ax.set(title='Доход — логарифмическая шкала',
                   xlabel='Годовой доход, логарифмическая шкала', ylabel='Количество записей')
income_log_ax.xaxis.set_major_formatter(money_format)
income_log_ax.legend()
plt.show()


**Выводы и решения по person_income**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f16', 'У дохода длинный правый хвост — медиана 55 000, встречаются редкие доходы свыше миллиона', 'Проверить, отличаются ли редкие высокие доходы по доле дефолтов, прежде чем решать, как их обрабатывать.\nЗаёмщики с высокими доходами могут существенно отличаться по кредитному поведению — эту гипотезу предстоит проверить'),
])
display(findings_view(conclusions, only_new=True))

##### 3.1.3.4 loan_intent


##### Цель кредита — loan_intent

Посмотрим, сколько заявок относится к каждой цели и какую долю они составляют

In [ ]:
intent_counts = data['loan_intent'].value_counts()
intent_table = pd.DataFrame({
    'Количество': intent_counts,
    'Доля, %': 100 * intent_counts / len(data),
})
display(intent_table)

intent_labels = {
    'EDUCATION': 'Образование',
    'MEDICAL': 'Лечение',
    'VENTURE': 'Бизнес',
    'PERSONAL': 'Личные нужды',
    'DEBTCONSOLIDATION': 'Рефинансирование долгов',
    'HOMEIMPROVEMENT': 'Ремонт жилья',
}
intent_names = [intent_labels[category] for category in intent_table.index]
intent_colors = sns.color_palette('Set2', len(intent_table))
fig, axes = plt.subplots(1, 2, figsize=(15, 5), **PLOT_LAYOUT)
bars = axes[0].barh(intent_names, intent_table['Доля, %'], color=intent_colors)
axes[0].invert_yaxis()
axes[0].bar_label(bars, labels=[
    fmt_count_share(count, share)
    for count, share in zip(intent_table['Количество'], intent_table['Доля, %'])
], padding=5, fontsize=9)
axes[0].set(title='Распределение по цели кредита',
            xlabel='Доля записей, %', ylabel='Цель кредита')
axes[0].set_xlim(0, intent_table['Доля, %'].max() * 1.45)

wedges, _, _ = axes[1].pie(intent_counts, colors=intent_colors, startangle=90,
                          autopct=fmt_percent,
                          pctdistance=0.72, wedgeprops={'edgecolor': 'white'})
axes[1].legend(wedges, intent_names, loc='center left',
               bbox_to_anchor=(1, 0.5), frameon=False)
axes[1].set_title('Доли целей кредита')
plt.show()


**Выводы и решения по loan_intent**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f17', 'Цели кредита представлены относительно равномерно — доли категорий от 11,09 % до 19,77 %', '—'),
])
display(findings_view(conclusions, only_new=True))

##### 3.1.3.5 person_emp_length


##### Стаж работы — person_emp_length

Посмотрим основные значения и разброс стажа

In [ ]:
employment = data['person_emp_length'].dropna()
employment_q1, employment_q3 = employment.quantile([0.25, 0.75])
employment_iqr = employment_q3 - employment_q1
employment_lower = employment_q1 - 1.5 * employment_iqr
employment_upper = employment_q3 + 1.5 * employment_iqr
employment_outliers = (employment < employment_lower) | (employment > employment_upper)

employment_stats = pd.concat([
    employment.describe(),
    pd.Series({
        'Пропусков': data['person_emp_length'].isna().sum(),
        'Коэффициент асимметрии': employment.skew(),
        'Верхняя граница по правилу 1,5 IQR': employment_upper,
        'Записей за границами 1,5 IQR': int(employment_outliers.sum()),
        'Доля записей за границами среди заполненных, %': 100 * employment_outliers.mean(),
    }),
]).to_frame('Значение')
display(employment_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), **PLOT_LAYOUT)
sns.histplot(x=employment, discrete=True, color=DISTRIBUTION_COLOR, ax=axes[0])
axes[0].axvline(employment.median(), color=MEDIAN_COLOR, linestyle='--',
               label=f'Медиана: {employment.median():g}')
axes[0].set(title='Распределение стажа', xlabel='Стаж, лет', ylabel='Количество записей')
axes[0].legend()
sns.boxplot(x=employment, color=BOX_COLOR, ax=axes[1])
axes[1].set(title='Стаж — ящик с усами', xlabel='Стаж, лет')
plt.show()


**Выводы и решения по person_emp_length**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f18', 'Небольшой стаж встречается чаще большого — медиана 4 года, наиболее частое значение 0 лет, большие значения встречаются реже', '—'),
])
display(findings_view(conclusions, only_new=True))

##### 3.1.3.6 loan_grade


##### Грейд кредита — loan_grade

Посмотрим, как часто встречаются грейды. Расположим их по порядку от A до G, а не по частоте

In [ ]:
grade_counts = data['loan_grade'].value_counts().reindex(list('ABCDEFG'), fill_value=0)
grade_table = pd.DataFrame({
    'Количество': grade_counts,
    'Доля, %': 100 * grade_counts / len(data),
})
display(grade_table)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), **PLOT_LAYOUT)
grade_colors = sns.color_palette('viridis', len(grade_table))
bars = axes[0].bar(grade_table.index, grade_table['Доля, %'], color=grade_colors)
axes[0].bar_label(bars, labels=[fmt_percent(share) for share in grade_table['Доля, %']],
                  padding=5, fontsize=9)
axes[0].set(title='Распределение грейдов', xlabel='Грейд займа', ylabel='Доля записей, %')
axes[0].set_ylim(0, grade_table['Доля, %'].max() * 1.18)

wedges, _, _ = axes[1].pie(
    grade_counts, colors=grade_colors, startangle=90,
    autopct=lambda share: fmt_percent(share, min_share=2),
    pctdistance=0.72, wedgeprops={'edgecolor': 'white'},
)
axes[1].legend(wedges, [
    f'{grade} — {fmt_percent(share)}' for grade, share in grade_table['Доля, %'].items()
], loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
axes[1].set_title('Доли грейдов')
plt.show()


**Выводы и решения по loan_grade**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f19', 'Грейды распределены неравномерно — A и B составляют 65,06 % записей, F и G вместе — 0,94 %', '—'),
])
display(findings_view(conclusions, only_new=True))

##### 3.1.3.7 loan_amnt


##### Сумма кредита — loan_amnt

Посмотрим, какие суммы встречаются чаще и насколько отличаются от них редкие крупные кредиты

In [ ]:
amount = data['loan_amnt'].dropna()
amount_q1, amount_q3 = amount.quantile([0.25, 0.75])
amount_iqr = amount_q3 - amount_q1
amount_lower = amount_q1 - 1.5 * amount_iqr
amount_upper = amount_q3 + 1.5 * amount_iqr
amount_outliers = (amount < amount_lower) | (amount > amount_upper)

amount_stats = pd.concat([
    amount.describe(),
    pd.Series({
        'Пропусков': data['loan_amnt'].isna().sum(),
        'Коэффициент асимметрии': amount.skew(),
        'Верхняя граница по правилу 1,5 IQR': amount_upper,
        'Записей за границами 1,5 IQR': int(amount_outliers.sum()),
        'Доля записей за границами среди заполненных, %': 100 * amount_outliers.mean(),
    }),
]).to_frame('Значение')
display(amount_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), **PLOT_LAYOUT)
sns.histplot(x=amount, binwidth=500, color=DISTRIBUTION_COLOR, ax=axes[0])
axes[0].axvline(amount.median(), color=MEDIAN_COLOR, linestyle='--',
               label=f'Медиана: {amount.median():g}')
axes[0].set(title='Распределение — сумма кредита', xlabel='Сумма кредита', ylabel='Количество записей')
axes[0].legend()
sns.boxplot(x=amount, color=BOX_COLOR, ax=axes[1])
axes[1].set(title='Сумма кредита — ящик с усами', xlabel='Сумма кредита')
for ax in axes:
    ax.xaxis.set_major_formatter(money_format)
plt.show()


**Выводы и решения по loan_amnt**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f20', 'Небольшие кредиты встречаются чаще крупных — медиана 8 000, наиболее частая сумма 10 000', '—'),
])
display(findings_view(conclusions, only_new=True))

##### 3.1.3.8 loan_int_rate


##### Процентная ставка — loan_int_rate

Посмотрим, как распределены ставки и есть ли несколько часто встречающихся диапазонов

In [ ]:
rate = data['loan_int_rate'].dropna()
rate_q1, rate_q3 = rate.quantile([0.25, 0.75])
rate_iqr = rate_q3 - rate_q1
rate_lower = rate_q1 - 1.5 * rate_iqr
rate_upper = rate_q3 + 1.5 * rate_iqr
rate_outliers = (rate < rate_lower) | (rate > rate_upper)

rate_stats = pd.concat([
    rate.describe(),
    pd.Series({
        'Пропусков': data['loan_int_rate'].isna().sum(),
        'Коэффициент асимметрии': rate.skew(),
        'Верхняя граница по правилу 1,5 IQR': rate_upper,
        'Записей за границами 1,5 IQR': int(rate_outliers.sum()),
        'Доля записей за границами среди заполненных, %': 100 * rate_outliers.mean(),
    }),
]).to_frame('Значение')
display(rate_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), **PLOT_LAYOUT)
sns.histplot(x=rate, binwidth=0.25, color=DISTRIBUTION_COLOR, ax=axes[0])
axes[0].axvline(rate.median(), color=MEDIAN_COLOR, linestyle='--',
               label=f'Медиана: {rate.median():g}')
axes[0].set(title='Распределение — процентная ставка', xlabel='Процентная ставка, % годовых', ylabel='Количество записей')
axes[0].legend()
sns.boxplot(x=rate, color=BOX_COLOR, ax=axes[1])
axes[1].set(title='Процентная ставка — ящик с усами', xlabel='Процентная ставка, % годовых')
plt.show()


**Выводы и решения по loan_int_rate**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f21', 'Распределение ставок имеет несколько пиков, медиана — 10,99 %, нет заметного перевеса в сторону более низких или более высоких значений', '—'),
])
display(findings_view(conclusions, only_new=True))

##### 3.1.3.9 loan_percent_income


##### Доля кредита от дохода — loan_percent_income

В данных это доля от 0 до 1, на графиках показываем проценты

In [ ]:
income_share = data['loan_percent_income'].dropna()
income_share_q1, income_share_q3 = income_share.quantile([0.25, 0.75])
income_share_iqr = income_share_q3 - income_share_q1
income_share_lower = income_share_q1 - 1.5 * income_share_iqr
income_share_upper = income_share_q3 + 1.5 * income_share_iqr
income_share_outliers = (income_share < income_share_lower) | (income_share > income_share_upper)

income_share_stats = pd.concat([
    income_share.describe(),
    pd.Series({
        'Коэффициент асимметрии': income_share.skew(),
        'Верхняя граница по правилу 1,5 IQR': income_share_upper,
        'Записей за границами 1,5 IQR': int(income_share_outliers.sum()),
        'Доля записей за границами, %': 100 * income_share_outliers.mean(),
    }),
]).to_frame('Значение')
display(income_share_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), **PLOT_LAYOUT)
sns.histplot(x=income_share, binwidth=0.01, color=DISTRIBUTION_COLOR, ax=axes[0])
axes[0].axvline(income_share.median(), color=MEDIAN_COLOR, linestyle='--',
               label=f'Медиана: {income_share.median():g}')
axes[0].set(title='Распределение — доля суммы кредита от дохода', xlabel='Доля суммы кредита от годового дохода, %', ylabel='Количество записей')
axes[0].legend()
sns.boxplot(x=income_share, color=BOX_COLOR, ax=axes[1])
axes[1].set(title='Доля суммы кредита от дохода — ящик с усами', xlabel='Доля суммы кредита от годового дохода, %')
for ax in axes:
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
plt.show()


**Выводы и решения по loan_percent_income**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f22', 'Небольшие доли кредита от дохода встречаются чаще крупных — медиана 15 %, большие доли встречаются реже', '—'),
])
display(findings_view(conclusions, only_new=True))

##### 3.1.3.10 cb_person_default_on_file


##### Прошлые дефолты — cb_person_default_on_file

N означает, что прошлых дефолтов не было, Y — что были. Посмотрим доли этих двух групп

In [ ]:
past_default_counts = data['cb_person_default_on_file'].value_counts().reindex(['N', 'Y'], fill_value=0)
past_default_table = pd.DataFrame({
    'Количество': past_default_counts,
    'Доля, %': 100 * past_default_counts / len(data),
})
display(past_default_table)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), **PLOT_LAYOUT)
past_default_labels = ['N — не было', 'Y — были']
past_default_colors = CATEGORY_COLORS[:2]
bars = axes[0].bar(past_default_labels, past_default_table['Доля, %'], color=past_default_colors)
axes[0].bar_label(bars, labels=[
    fmt_count_share(count, share)
    for count, share in zip(past_default_table['Количество'], past_default_table['Доля, %'])
], padding=5)
axes[0].set(title='Прошлые дефолты', xlabel='Дефолты в кредитной истории', ylabel='Доля записей, %')
axes[0].set_ylim(0, 100)
axes[1].pie(past_default_counts, labels=past_default_labels, colors=past_default_colors,
            startangle=90, autopct=fmt_percent,
            wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Доли заёмщиков по прошлым дефолтам')
plt.show()


**Выводы и решения по cb_person_default_on_file**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f23', 'У большинства заёмщиков нет зарегистрированных прошлых дефолтов — N составляет 82,32 %, Y — 17,68 %', '—'),
])
display(findings_view(conclusions, only_new=True))

##### 3.1.3.11 cb_person_cred_hist_length


##### Длина кредитной истории — cb_person_cred_hist_length

Посмотрим, сколько лет кредитной истории обычно есть у заёмщиков

In [ ]:
credit_history = data['cb_person_cred_hist_length'].dropna()
credit_history_q1, credit_history_q3 = credit_history.quantile([0.25, 0.75])
credit_history_iqr = credit_history_q3 - credit_history_q1
credit_history_lower = credit_history_q1 - 1.5 * credit_history_iqr
credit_history_upper = credit_history_q3 + 1.5 * credit_history_iqr
credit_history_outliers = (credit_history < credit_history_lower) | (credit_history > credit_history_upper)

credit_history_stats = pd.concat([
    credit_history.describe(),
    pd.Series({
        'Коэффициент асимметрии': credit_history.skew(),
        'Верхняя граница по правилу 1,5 IQR': credit_history_upper,
        'Записей за границами 1,5 IQR': int(credit_history_outliers.sum()),
        'Доля записей за границами, %': 100 * credit_history_outliers.mean(),
    }),
]).to_frame('Значение')
display(credit_history_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), **PLOT_LAYOUT)
sns.histplot(x=credit_history, discrete=True, color=DISTRIBUTION_COLOR, ax=axes[0])
axes[0].axvline(credit_history.median(), color=MEDIAN_COLOR, linestyle='--',
               label=f'Медиана: {credit_history.median():g}')
axes[0].set(title='Распределение — длина кредитной истории', xlabel='Длина кредитной истории, лет', ylabel='Количество записей')
axes[0].legend()
sns.boxplot(x=credit_history, color=BOX_COLOR, ax=axes[1])
axes[1].set(title='Длина кредитной истории — ящик с усами', xlabel='Длина кредитной истории, лет')
plt.show()


**Выводы и решения по cb_person_cred_hist_length**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f24', 'Преобладает короткая кредитная история — медиана 4 года.\nРаспределение ступенчатое: внутри отдельных диапазонов частоты близки, между диапазонами резко меняются.\nЭто может отражать особенности отбора или сборки данных по группам либо искусственное формирование признака', 'Проверить связь длины кредитной истории с возрастом — тепловая карта сочетаний значений.\nПроверить, совпадают ли группы с последовательными участками файла.\nСравнить между группами доход, грейд, ставку и доли пропусков.\nПроверить связь с loan_status, сравнивая группы также внутри одинаковых возрастов и грейдов'),
])
display(findings_view(conclusions, only_new=True))

##### 3.1.3.12 loan_status


##### Результат кредита — loan_status

Посмотрим число и долю дефолтов, чтобы оценить дисбаланс классов

In [ ]:
target_counts = data['loan_status'].value_counts().reindex([0, 1], fill_value=0)
target_table = pd.DataFrame({
    'Количество': target_counts,
    'Доля, %': 100 * target_counts / len(data),
})
display(target_table)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), **PLOT_LAYOUT)
target_labels = ['0 — без дефолта', '1 — дефолт']
target_colors = CATEGORY_COLORS[:2]
bars = axes[0].bar(target_labels, target_table['Доля, %'], color=target_colors)
axes[0].bar_label(bars, labels=[
    fmt_count_share(count, share)
    for count, share in zip(target_table['Количество'], target_table['Доля, %'])
], padding=5)
axes[0].set(title='Баланс классов loan_status', xlabel='Исход кредита', ylabel='Доля записей, %')
axes[0].set_ylim(0, 100)
axes[1].pie(target_counts, labels=target_labels, colors=target_colors,
            startangle=90, autopct=fmt_percent,
            wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Доли исходов кредита')
plt.show()


**Выводы и решения по loan_status**

In [ ]:
conclusions = update_findings(conclusions, [
    ('f25', 'Классы несбалансированы — 78,13 % кредитов без дефолта и 21,87 % с дефолтом, соотношение примерно 3,57 к 1', '—'),
])
display(findings_view(conclusions, only_new=True))

#### Выводы и решения после базового EDA

In [ ]:
conclusions = update_findings(conclusions)
display(findings_view(conclusions))


### 3.2 Разбиение данных

**Входы**

- Таблица после очистки

**Выходы**

- train, validation и test

**Что делаем**

Разделим записи на три части. Сохраним в них примерно одинаковую долю дефолтов

##### Зачем делить данные на три части

Базовый обзор, проверка ошибок и анализ распределений завершены. Дальше будем изучать связи между признаками и с `loan_status`, проверять гипотезы и на этой основе выбирать признаки и обработку. Поэтому делим данные именно сейчас: эти решения принимаем по train, чтобы validation и test не влияли на устройство модели\
validation оставляем для выбора модели и порога, test — для итоговой оценки

Решения о дальнейших изменениях, включая отбор признаков, предобработку, принимаем только на основе train, а производим со всеми треми наборами, ведь модель будем применять ко всем

##### Как делим данные

Выделим 70% записей для train, оставшиеся 30% поделим поровну между validation и test\
В каждой части сохраним примерно одинаковую долю дефолтов. Такое разбиение называют **стратифицированным**. `random_state=42` фиксирует случайный выбор, чтобы при повторном запуске получить те же части

In [ ]:
from sklearn.model_selection import train_test_split

train, remaining = train_test_split(
    data,
    test_size=0.30,
    stratify=data['loan_status'],
    random_state=42,
)

validation, test = train_test_split(
    remaining,
    test_size=0.50,
    stratify=remaining['loan_status'],
    random_state=42,
)


In [ ]:
conclusions = update_findings(conclusions, [
    ('split', f'После разбиения: train — {len(train)} записей, validation — {len(validation)}, test — {len(test)}', 'Связи изучаем на train, модель и порог выбираем по validation, итог проверяем на test'),
])
display(findings_view(conclusions))

### 3.3 Корреляционный анализ на train

**Входы**

- train и первые наблюдения

**Выходы**

- Матрица связей между переменными
- Обновлённые выводы и предположения

**Что делаем**

На train исследуем все пары переменных, включая пары с `loan_status`. Сначала приведём таблицу гипотез из предыдущих выводов, затем рассчитаем полную матрицу связей

#### 3.3.1 Гипотезы из предыдущих выводов

**Входы**

- train и наблюдения из EDA

**Выходы**

- Предположения и способы их проверки

**Что делаем**

Соберём гипотезы из предыдущих выводов и укажем способы их проверки

##### Посмотрим, какие переменные связаны

На train оценим все попарные связи, включая связи с целевой переменной `loan_status`. Меры выберем по типам переменных\
Гипотезы о совместных зависимостях трёх и более переменных и форме связей проверим отдельно, в подэтапе 3.4

##### Таблица гипотез

Ниже собраны гипотезы из предыдущих наблюдений и способы их проверки. Среди них есть и двухмерные гипотезы — о связи двух переменных\
Попарный анализ не ограничивается этой таблицей: проверим все пары переменных, а не только перечисленные

In [ ]:
conclusions = update_findings(conclusions, [
    ('f16', 'У дохода длинный правый хвост — медиана 55 000, встречаются редкие доходы свыше миллиона', 'Проверить, отличаются ли редкие высокие доходы по доле дефолтов, прежде чем решать, как их обрабатывать'),
    ('f24', 'Преобладает короткая кредитная история — медиана 4 года.\nРаспределение ступенчатое: внутри отдельных диапазонов частоты близки, между диапазонами резко меняются.\nГипотеза: это может отражать особенности отбора или сборки данных по группам либо искусственное формирование признака', 'Проверить, совпадают ли группы с последовательными участками файла.\nСравнить между группами доход, грейд, ставку и доли пропусков'),
    ('f27', 'Гипотеза: чем хуже грейд кредита loan_grade, тем выше процентная ставка loan_int_rate', 'Проверить на этапе попарного анализа — Spearman'),
    ('f28', 'Гипотеза: возраст положительно связан с длиной кредитной истории', 'Проверить на этапе попарного анализа — Pearson'),
    ('f29', 'Гипотеза: возраст положительно связан с трудовым стажем', 'Проверить на этапе попарного анализа — Pearson'),
    ('f30', 'Гипотеза: цель кредита связана с loan_status', 'Проверить на этапе попарного анализа — Cramér’s V'),
    ('f31', 'Гипотеза: loan_percent_income ≈ loan_amnt / person_income.\nРанее выявленные расхождения не позволяют считать формулу выполненной для всех записей', 'Сравнить долю с отношением суммы к доходу на train и посмотреть на расхождения'),
    ('f32', 'Гипотеза: связь прошлых дефолтов с loan_status зависит от грейда кредита.\nПока это предположение; его причины ещё нужно выяснить', 'Сравнить доли дефолтов с прошлым дефолтом и без него внутри каждого грейда'),
    ('f33', 'Гипотеза: вероятность дефолта увеличивается с ростом отношения суммы кредита к доходу', 'Сравнить доли дефолтов для разных сочетаний дохода и суммы кредита'),
    ('f34', 'Гипотеза: грейд кредита и ставка совместно связаны с вероятностью дефолта, часть их информации может быть избыточной', 'Сравнить ставки и доли дефолтов внутри грейдов'),
    ('f35', 'Гипотеза: длина кредитной истории связана с loan_status при одинаковом возрасте заёмщиков', 'Сравнить длину истории и долю дефолтов внутри возрастных групп и грейдов'),
    ('f36', 'Гипотеза: заёмщики с высокими доходами отличаются по кредитному поведению', 'Отдельно посмотреть на дефолты среди заёмщиков с высокими доходами'),
    ('f37', 'Гипотеза: связь грейда кредита и процентной ставки имеет ступенчатую форму', 'Посмотреть, как меняются ставки от грейда к грейду'),
    ('f38', 'Гипотеза: связь возраста и длины кредитной истории нелинейна', 'Построить тепловую карту возраста и длины истории: цвет покажет, какие сочетания встречаются чаще'),
])
display(findings_view(conclusions))

#### 3.3.2 Матрица связей

**Входы**

- train и типы переменных

**Выходы**

- Коэффициенты связи для всех пар переменных

**Что делаем**

Для разных типов данных выберем подходящую меру связи и соберём результаты в матрицу, включая пары с loan_status

##### Как измеряем связи

Одна мера не подходит сразу для чисел, неупорядоченных категорий и грейдов, поэтому выбираем метод по типу пары

| Тип переменной | Числовой | Порядковый | Номинальный |
| --- | --- | --- | --- |
| **Числовой** | Пирсон $r$ | Спирмен $\rho$ | Нормализованная взаимная информация |
| **Порядковый** | Спирмен $\rho$ | Спирмен $\rho$ | Краскел — Уоллис $\varepsilon^2$ |
| **Номинальный** | Нормализованная взаимная информация | Краскел — Уоллис $\varepsilon^2$ | V Крамера |

Бинарные столбцы считаем номинальными. У грейда сохраняем порядок A < B < C < D < E < F < G

Для каждой пары используем строки без пропусков в её двух столбцах. Соберём коэффициенты в одну матрицу

###### Общая матрица

Зададим типы признаков и подготовим матрицу для коэффициентов связи

In [ ]:
from itertools import combinations_with_replacement
from scipy.stats import chi2_contingency, kruskal
from sklearn.feature_selection import mutual_info_classif


# Общая матрица: инициализируем один раз перед последовательными этапами
association_columns = train.columns
association_matrix = pd.DataFrame(
    np.nan, index=association_columns, columns=association_columns, dtype=float,
)

# Для выбора мер связи используем смысловые типы из описаний полей.
association_types = pd.Series({
    'person_age': 'числовой',
    'person_income': 'числовой',
    'person_home_ownership': 'номинальный',
    'person_emp_length': 'числовой',
    'loan_intent': 'номинальный',
    'loan_grade': 'порядковый',
    'loan_amnt': 'числовой',
    'loan_int_rate': 'числовой',
    'loan_status': 'бинарный',
    'loan_percent_income': 'числовой',
    'cb_person_default_on_file': 'бинарный',
    'cb_person_cred_hist_length': 'числовой',
}).reindex(association_columns)
numerical_columns = association_types.index[association_types.eq('числовой')].tolist()


###### Числовые пары — Pearson

Коэффициент Pearson измеряет силу и направление линейной связи, принимает значения от −1 до 1

In [ ]:
# Числовые пары
pearson_matrix = train[numerical_columns].corr(method='pearson', min_periods=2)
association_matrix.loc[numerical_columns, numerical_columns] = pearson_matrix


###### Номинальные пары — Cramér’s V

Cramér’s V рассчитываем по таблице сопряжённости и статистике хи-квадрат без коррекции смещения. Диапазон — от 0 до 1, бинарные признаки включаем в номинальные

In [ ]:
def cramers_v(left, right):
    paired = pd.concat([left, right], axis=1).dropna()
    if paired.empty:
        return np.nan
    table = pd.crosstab(paired.iloc[:, 0], paired.iloc[:, 1])
    # Удаляем ненаблюдаемые категории, если столбцы имеют dtype category
    table = table.loc[table.sum(axis=1).gt(0), table.sum(axis=0).gt(0)]
    k, m = table.shape
    if min(k, m) < 2:
        return np.nan
    chi2 = chi2_contingency(table, correction=False)[0]
    return np.sqrt(chi2 / (table.to_numpy().sum() * min(k - 1, m - 1)))


nominal_columns = association_types.index[
    association_types.isin(['номинальный', 'бинарный'])
].tolist()

# Номинальные пары, включая бинарные
for left, right in combinations_with_replacement(nominal_columns, 2):
    value = cramers_v(train[left], train[right])
    association_matrix.loc[left, right] = value
    association_matrix.loc[right, left] = value


###### Числовые и порядковые пары — Spearman

Spearman оценивает монотонную связь по рангам с усреднением совпадений. Диапазон — от −1 до 1. Для loan_grade задаём порядок A < B < C < D < E < F < G; для порядковых пар используем ту же меру

In [ ]:
# Явный порядок категорий, без изменения исходного train
ordinal_orders = {'loan_grade': ['A', 'B', 'C', 'D', 'E', 'F', 'G']}
ordinal_columns = association_types.index[association_types.eq('порядковый')].tolist()

for ordinal in ordinal_columns:
    order = ordinal_orders[ordinal]
    observed = train[ordinal].dropna()
    if not observed.isin(order).all():
        raise ValueError(f'Неизвестные категории в {ordinal}')
    ordinal_values = train[ordinal].map({grade: rank for rank, grade in enumerate(order)})
    for numerical in numerical_columns:
        paired = pd.DataFrame({
            'numerical': train[numerical],
            'ordinal': ordinal_values,
        }).dropna()
        value = paired.corr(method='spearman', min_periods=2).loc['numerical', 'ordinal']
        association_matrix.loc[numerical, ordinal] = value
        association_matrix.loc[ordinal, numerical] = value

# Пары порядковых переменных, включая диагональ
ordinal_data = pd.DataFrame({
    column: train[column].map({value: rank for rank, value in enumerate(ordinal_orders[column])})
    for column in ordinal_columns
}, index=train.index)
association_matrix.loc[ordinal_columns, ordinal_columns] = ordinal_data.corr(
    method='spearman', min_periods=2,
)


###### Числовые и номинальные пары — NMI

Нормализуем оценку взаимной информации по энтропии номинального признака: $I(X;Y)/H(Y)$. Мера учитывает различия средних, разбросов и форм распределений. При нулевой энтропии нормализация не определена

In [ ]:
def numerical_nominal_nmi(numerical, nominal):
    paired = pd.DataFrame({'x': numerical, 'y': nominal}).dropna()
    if paired.empty:
        return np.nan, np.nan, np.nan
    labels, _ = pd.factorize(paired['y'], sort=False)
    counts = np.bincount(labels)
    probabilities = counts / counts.sum()
    entropy = float(-(probabilities * np.log(probabilities)).sum())
    if entropy <= 0:
        return np.nan, entropy, np.nan
    if paired['x'].nunique() <= 1:
        return 0.0, entropy, 0.0
    if counts.min() < 2:
        return np.nan, entropy, np.nan
    mi = mutual_info_classif(
        paired[['x']], labels, discrete_features=False,
        n_neighbors=3, random_state=42,
    )[0]
    ratio = mi / entropy
    return float(mi), entropy, float(np.clip(ratio, 0.0, 1.0))


###### Заполнение числовых × номинальных пар

Оцениваем взаимную информацию без разбиения числового признака на категории. Нормализованные оценки ограничиваем интервалом от 0 до 1

In [ ]:
# Числовые × номинальные
nmi_records = []
for numerical in numerical_columns:
    for nominal in nominal_columns:
        mi, entropy, value = numerical_nominal_nmi(train[numerical], train[nominal])
        association_matrix.loc[numerical, nominal] = value
        association_matrix.loc[nominal, numerical] = value
        nmi_records.append({
            'Numerical': numerical, 'Nominal': nominal,
            'MI estimate': mi, 'H(Y)': entropy,
            'MI / H(Y)': mi / entropy if entropy > 0 else np.nan,
            'NMI': value,
        })

nmi_estimates = pd.DataFrame(nmi_records).set_index(['Numerical', 'Nominal'])


###### Порядковые и номинальные пары — Краскел — Уоллис

Критерий сравнивает ранги между группами, учитывая совпадения. Он использует порядок категорий без предположения о равных расстояниях между ними. В матрицу записываем размер эффекта

$$\varepsilon^2=\frac{H-k+1}{n-k},$$

где $k$ — число групп, $n$ — число наблюдений. Метод может пропустить зависимость, если распределения различаются, а средние ранги групп совпадают

In [ ]:
def ordinal_nominal_kruskal(ordinal, nominal, order):
    paired = pd.DataFrame({'ordinal': ordinal, 'nominal': nominal}).dropna()
    paired['rank_code'] = paired['ordinal'].map(
        {category: rank for rank, category in enumerate(order)}
    )
    groups = [
        group['rank_code'].to_numpy(dtype=float)
        for _, group in paired.groupby('nominal', observed=True, sort=False)
    ]
    k, n = len(groups), len(paired)
    if k < 2 or n <= k or paired['rank_code'].nunique() < 2:
        return np.nan, k, n, np.nan
    h = float(kruskal(*groups).statistic)
    effect = (h - k + 1) / (n - k)
    return h, k, n, float(effect)


###### Заполнение порядковых × номинальных пар

Сохраняем исходную оценку ε². Только при выборе цвета ограничиваем её интервалом от 0 до 1, поскольку выборочная оценка может быть отрицательной

In [ ]:
# Порядковые × номинальные
kruskal_records = []
for ordinal in ordinal_columns:
    for nominal in nominal_columns:
        h, k, n, effect = ordinal_nominal_kruskal(
            train[ordinal], train[nominal], ordinal_orders[ordinal],
        )
        association_matrix.loc[ordinal, nominal] = effect
        association_matrix.loc[nominal, ordinal] = effect
        kruskal_records.append({
            'Ordinal': ordinal, 'Nominal': nominal,
            'H': h, 'k': k, 'n': n, 'epsilon_squared': effect,
        })

kruskal_effects = pd.DataFrame(kruskal_records).set_index(['Ordinal', 'Nominal'])


###### Как читать матрицу

Каждая ячейка показывает связь двух переменных. Смотрим только нижнюю половину: верхняя повторяла бы те же пары\
Чем насыщеннее цвет, тем сильнее связь по выбранной мере. У Pearson и Spearman знак в подписи показывает направление: плюс — значения в среднем растут вместе, минус — меняются в противоположных направлениях. Для остальных мер направления нет\
Палитры различают методы; числа разных методов напрямую не сравниваем. Пустая ячейка означает, что коэффициент не удалось определить

In [ ]:
def plot_association_matrix(matrix, title):
    # Разные палитры обозначают разные меры, без общей цветовой шкалы
    pearson_mask = pd.DataFrame(True, index=matrix.index, columns=matrix.columns)
    pearson_mask.loc[numerical_columns, numerical_columns] = False
    ordinal = association_types.index[association_types.eq('порядковый')]
    pearson_mask.loc[numerical_columns, ordinal] = False
    pearson_mask.loc[ordinal, numerical_columns] = False
    pearson_mask.loc[ordinal, ordinal] = False
    # Скрываем верхний треугольник только при отображении
    upper_mask = np.triu(np.ones(matrix.shape, dtype=bool), k=1)
    fig, ax = plt.subplots(figsize=(14, 12), constrained_layout=True)
    sns.heatmap(
        matrix, mask=matrix.isna() | pearson_mask | upper_mask,
        cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        annot=True, fmt='.2f', square=True, cbar=False,
        linewidths=0, ax=ax,
    )
    nominal_mask = pd.DataFrame(True, index=matrix.index, columns=matrix.columns)
    nominal = association_types.index[association_types.isin(['номинальный', 'бинарный'])]
    nominal_mask.loc[nominal, nominal] = False
    if (matrix.notna() & ~nominal_mask).any().any():
        sns.heatmap(
            matrix, mask=matrix.isna() | nominal_mask | upper_mask,
            cmap='Purples', vmin=0, vmax=1,
            annot=True, fmt='.2f', square=True, cbar=False,
            linewidths=0, ax=ax,
        )
    nmi_mask = pd.DataFrame(True, index=matrix.index, columns=matrix.columns)
    nmi_mask.loc[numerical_columns, nominal] = False
    nmi_mask.loc[nominal, numerical_columns] = False
    if (matrix.notna() & ~nmi_mask).any().any():
        sns.heatmap(
            matrix, mask=matrix.isna() | nmi_mask | upper_mask,
            cmap='Greens', vmin=0, vmax=1,
            annot=True, fmt='.2f', square=True, cbar=False,
            linewidths=0, ax=ax,
        )
    kw_mask = pd.DataFrame(True, index=matrix.index, columns=matrix.columns)
    kw_mask.loc[ordinal, nominal] = False
    kw_mask.loc[nominal, ordinal] = False
    if (matrix.notna() & ~kw_mask).any().any():
        sns.heatmap(
            matrix.clip(lower=0, upper=1),
            mask=matrix.isna() | kw_mask | upper_mask,
            cmap='Oranges', vmin=0, vmax=1,
            annot=matrix, fmt='.2f', square=True, cbar=False,
            linewidths=0, ax=ax,
        )
    ax.xaxis.tick_bottom()
    ax.tick_params(axis='x', labeltop=False, labelbottom=True)
    plt.setp(ax.get_xticklabels(), rotation=60, ha='right', rotation_mode='anchor')
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.set_title(title, pad=15)
    ax.set_xlabel(
        'Pearson / Spearman: синий (−), красный (+), интенсивность по |r| / |ρ|\n'
        "Cramér’s V: фиолетовый, интенсивность по V; NMI: зелёный, интенсивность по NMI\nKruskal–Wallis: оранжевый, интенсивность по ε² (в пределах [0, 1])\nПустые ячейки — NaN",
        labelpad=15,
    )
    plt.show()


plot_association_matrix(
    association_matrix,
    'Матрица попарных ассоциаций, train',
)


#### Что показал попарный анализ

Сопоставим гипотезы с результатами на train — 22 687 записях. Для пар с пропусками использовано меньше строк\
Обновим таблицу выводов. Вопросы о сочетаниях нескольких признаков проверим на следующем шаге

In [ ]:
conclusions = update_findings(conclusions, [
    ('f39', 'На train между loan_grade и loan_int_rate наблюдается сильная связь с одним направлением изменения — Spearman ρ = 0,947.\nХудшему грейду соответствует более высокая ставка', '—'),
    ('f40', 'Возраст и длина кредитной истории имеют сильную положительную линейную связь — Pearson r = 0,879.\nЭто не объясняет ступенчатое распределение длины истории', '—'),
    ('f41', 'Возраст и стаж имеют небольшую положительную линейную корреляцию — Pearson r = 0,171', '—'),
    ('f42', 'Между целью кредита и loan_status наблюдается ассоциация — Cramér’s V = 0,125.\nКоэффициент не указывает, какие цели связаны с большей долей дефолтов', '—'),
    ('f43', 'Для суммы кредита и доли кредита от дохода Pearson r = 0,577, для дохода и этой доли r = −0,294', '—'),
    ('f44', 'Прошлый дефолт связан с процентной ставкой по оценке NMI = 0,470, с грейдом кредита — по ε² = 0,296.\nМетоды отражают разные виды связи, их значения напрямую не сравниваем', '—'),
    ('f45', 'Для loan_status и типа жилья Cramér’s V = 0,243, для loan_status и прошлого дефолта V = 0,178.\nКоэффициенты показывают связь, но не говорят, что одно вызывает другое', '—'),
    ('f46', 'Для числовых признаков и loan_status получены оценки NMI: доход — 0,210, доля кредита от дохода — 0,156, ставка — 0,143, сумма — 0,036.\nКаждое число описывает один признак отдельно. Эти оценки нельзя складывать или считать готовой оценкой важности признаков в модели', '—'),
    ('f47', 'Между loan_grade и loan_status получен размер эффекта Краскела — Уоллиса ε² = 0,109.\nОн показывает, насколько грейды различаются между двумя исходами кредита, но не показывает направление связи', '—'),
    ('f48', 'Подпись 0.00 ещё не означает, что связи нет.\nНапример, для дохода и ставки Pearson r = 0,001817, для грейда и цели кредита ε² = 0,000542 — значения округлены.\nНоль у NMI тоже не доказывает отсутствия связи: отрицательные оценки, возникшие из-за погрешности расчёта по выборке, заменяются нулём', '—'),
], remove=['f27', 'f28', 'f29', 'f30'])
display(findings_view(conclusions))

### 3.4 Проверка гипотез на train

**Входы**

- train, предыдущие выводы и сформулированные гипотезы

**Выходы**

- Результаты проверки гипотез и обновлённая таблица выводов

**Что делаем**

Проверим гипотезы из предыдущих выводов о зависимостях трёх и более переменных. Сравним связи внутри групп и отдельно рассмотрим форму двухмерных зависимостей, которую не описывает один попарный коэффициент

#### Подготовка групп

Группы для проверок сформируем по train

In [ ]:
analysis_train = train.copy()
analysis_train['grade_ordered'] = pd.Categorical(analysis_train['loan_grade'], categories=list('ABCDEFG'), ordered=True)
analysis_train['income_quartile'] = pd.qcut(analysis_train['person_income'], 4, duplicates='drop')
analysis_train['amount_quartile'] = pd.qcut(analysis_train['loan_amnt'], 4, duplicates='drop')
analysis_train['age_band'] = pd.cut(analysis_train['person_age'], [18, 25, 35, 50, 100], include_lowest=True)
analysis_train['history_band'] = pd.cut(analysis_train['cb_person_cred_hist_length'], [0, 5, 10, 15, 20, 40], include_lowest=True)
analysis_train['file_block'] = pd.cut(analysis_train.index, [0, 8000, 16000, 24000, len(data_raw)], include_lowest=True)


def default_summary(frame, groups):
    return frame.groupby(groups, observed=True)['loan_status'].agg(n='size', defaults='sum', default_rate='mean')

def group_summary_view(table):
    return table.round(4).rename(columns={
        'n': 'Записей', 'defaults': 'Дефолтов', 'default_rate': 'Доля дефолтов',
        'rate_n': 'Известных ставок', 'rate_median': 'Медиана ставки',
        'rate_q25': 'Ставка: 25%', 'rate_q75': 'Ставка: 75%',
        'history_groups': 'Групп по длине истории', 'min_cell_n': 'Минимум записей в группе',
        'min_default_rate': 'Наименьшая доля дефолтов', 'max_default_rate': 'Наибольшая доля дефолтов',
        'age_median': 'Медиана возраста', 'history_median': 'Медиана длины истории',
        'income_median': 'Медиана дохода', 'grade_A_share': 'Доля грейда A',
        'rate_missing': 'Доля пропусков ставки',
    })


#### 3.4.1 Грейд и ставка

Сравним распределение ставки и частоту дефолта по грейдам. Дополнительно разобьём ставки внутри каждого грейда на три группы

In [ ]:
grade_summary = analysis_train.groupby('grade_ordered', observed=True).agg(
    n=('loan_status', 'size'), default_rate=('loan_status', 'mean'),
    rate_n=('loan_int_rate', 'count'), rate_median=('loan_int_rate', 'median'),
    rate_q25=('loan_int_rate', lambda s: s.quantile(.25)),
    rate_q75=('loan_int_rate', lambda s: s.quantile(.75)),
)
# Внутри каждого грейда — свои терцили ставки: масштабы между грейдами не смешиваем.
analysis_train['rate_within_grade'] = analysis_train.groupby('grade_ordered', observed=True)['loan_int_rate'].transform(
    lambda s: pd.qcut(s, 3, labels=False, duplicates='drop')
)
within_grade_rate_summary = default_summary(analysis_train.dropna(subset=['rate_within_grade']), ['grade_ordered', 'rate_within_grade'])
display(group_summary_view(grade_summary))
display(group_summary_view(within_grade_rate_summary))


#### 3.4.2 Доля кредита от дохода

Проверим, насколько исходная доля соответствует отношению суммы кредита к доходу

In [ ]:
ratio_error = (analysis_train['loan_percent_income'] - analysis_train['loan_amnt'] / analysis_train['person_income']).abs()
ratio_summary = pd.DataFrame({
    'Наблюдений': [len(analysis_train)],
    'Расхождений > 0,005': [int((ratio_error > .005 + 1e-9).sum())],
    'Медиана абсолютного расхождения': [ratio_error.median()],
    'Максимальное расхождение': [ratio_error.max()],
})
display(ratio_summary.round(4))


#### 3.4.3 Прошлые дефолты и грейд

Сравним частоты исхода по прошлым дефолтам внутри каждого грейда

In [ ]:
interaction_summary = default_summary(analysis_train, ['grade_ordered', 'cb_person_default_on_file'])
display(group_summary_view(interaction_summary))


#### 3.4.4 Сумма кредита и доход

Сравним частоты дефолта для сочетаний четырёх групп дохода и суммы

In [ ]:
amount_income_summary = default_summary(analysis_train, ['income_quartile', 'amount_quartile'])
display(group_summary_view(amount_income_summary))


#### 3.4.5 Кредитная история при сопоставимом возрасте и грейде

Сравним группы по длине истории внутри возрастных диапазонов и грейдов. В условной сводке оставим группы от 30 записей

In [ ]:
age_history_summary = default_summary(analysis_train, ['age_band', 'history_band'])
age_grade_history_summary = default_summary(analysis_train, ['age_band', 'grade_ordered', 'history_band'])
conditional_history_checks = (age_grade_history_summary.loc[age_grade_history_summary['n'] >= 30]
    .groupby(['age_band', 'grade_ordered'], observed=True)
    .agg(history_groups=('n', 'size'), min_cell_n=('n', 'min'),
         min_default_rate=('default_rate', 'min'), max_default_rate=('default_rate', 'max')))

display(group_summary_view(age_history_summary))
display(group_summary_view(conditional_history_checks))


#### 3.4.6 Высокие доходы

Граница высокого дохода — 95-й процентиль train. Сравним частоты дефолта ниже и выше неё

In [ ]:
income_95 = float(analysis_train['person_income'].quantile(.95))
high_income_summary = default_summary(analysis_train.assign(high_income=analysis_train['person_income'] > income_95), ['high_income'])
display(pd.Series({'Граница высокого дохода': income_95}))
display(group_summary_view(high_income_summary))


#### 3.4.7 Порядок записей в файле

Проверим, меняются ли состав заёмщиков и пропуски между последовательными блоками файла. Номер строки в признаки модели не включаем

In [ ]:
file_summary = analysis_train.groupby('file_block', observed=True).agg(
    n=('loan_status', 'size'), age_median=('person_age', 'median'),
    history_median=('cb_person_cred_hist_length', 'median'), income_median=('person_income', 'median'),
    grade_A_share=('loan_grade', lambda s: s.eq('A').mean()),
    rate_missing=('loan_int_rate', lambda s: s.isna().mean()),
)

display(group_summary_view(file_summary))


#### 3.4.8 Форма связей

Посмотрим на ставки по грейдам и сочетания возраста с длиной кредитной истории

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(data=analysis_train, x='grade_ordered', y='loan_int_rate', ax=axes[0], color='#78a6c8', showfliers=False)
axes[0].set(title='Ставка по грейду (train)', xlabel='Грейд A–G', ylabel='Процентная ставка, %')
age_history_counts = pd.crosstab(analysis_train['cb_person_cred_hist_length'], analysis_train['person_age'])
sns.heatmap(np.log1p(age_history_counts), cmap='Blues', ax=axes[1], cbar_kws={'label': 'log(1 + число наблюдений)'})
axes[1].set(title='Возраст и кредитная история (train)', xlabel='Возраст', ylabel='Длина кредитной истории')
plt.tight_layout()
plt.show()


#### Выводы по проверке гипотез

In [ ]:
# Дополняем общую таблицу результатами проверок на train
_analysis_updates = [
    ('grade_rate', f"Медианы ставки по грейдам A–G: {grade_summary['rate_median'].round(2).to_dict()}. Межквартильные интервалы приведены в таблице",
     'Ставки растут от грейда к грейду, но по одному грейду точную ставку не восстановить. Оставляем оба признака'),
    ('ratio', f"На train расхождений доли кредита с суммой/доходом >0,005: {int((ratio_error > .005 + 1e-9).sum())} из {len(train)}",
     'Точное равенство выполняется не везде. Причина расхождений неизвестна, поэтому исходную долю не заменяем'),
    ('income_amount', 'Сравнили доли дефолтов в группах по доходу и сумме кредита; рядом указали число записей',
     'Видим, как различаются группы, но не устанавливаем причину различий. Проверим более гибкие модели на validation'),
    ('default_grade', 'Сопоставлены частоты дефолта Y/N внутри грейдов; в редких грейдах наблюдений мало',
     'По этим долям нельзя уверенно сказать, что связь прошлого дефолта с новым исходом меняется именно из-за грейда. Вопрос остаётся открытым'),
    ('history_conditional', 'Сравнили длину истории и доли дефолтов внутри возрастных групп и грейдов',
     'По таким группам ещё нельзя отделить вклад истории от остальных различий между заёмщиками. В модели признак оставляем'),
    ('high_income', f"Порог высокого дохода — 95-й процентиль train ({income_95:,.0f}); доли дефолтов ниже/выше: {high_income_summary['default_rate'].round(4).to_dict()}",
     'Это сравнение наших групп. Оно не даёт основания удалять высокие доходы или ожидать тех же долей в других данных'),
    ('file_order', f"Медианы кредитной истории по последовательным блокам файла: {file_summary['history_median'].to_list()}",
     'В разных частях файла состав записей отличается. Почему — не знаем; номер строки в модель не подаём'),
    ('age_history', 'Тепловая карта показывает ограниченные возрастом области сочетаний с длиной истории; Pearson r=0,879 не описывает всю форму',
     'Попробуем модели на деревьях, которые допускают более сложные связи. По одной картине нельзя установить, как автор собирал данные'),
]
_aliases = {
    'grade_rate': 'grade_rate_details', 'ratio': 'f12',
    'income_amount': 'f33', 'default_grade': 'f32',
    'history_conditional': 'f35', 'high_income': 'high_income',
    'file_order': 'file_order', 'age_history': 'age_history_shape',
}
_additions = [(_aliases[key], finding, decision) for key, finding, decision in _analysis_updates]
_additions.append((
    'f34', 'Сравнили доли дефолтов по ставке внутри каждого грейда',
    'Оставляем грейд и ставку для сравнения моделей. Их отдельную полезность этим сравнением не установили',
))
_additions.append(('f16', conclusions.loc['f16', 'Выводы'], 'Высокие доходы не удаляем; доли дефолтов для них сравнены отдельно'))
_additions.append(('f24', conclusions.loc['f24', 'Выводы'].split('.\nГипотеза:')[0].strip(),
                   'Связь с возрастом и порядком файла проверена; происхождение данных не установлено'))
conclusions = update_findings(conclusions, _additions, remove=['f31', 'f36', 'f37', 'f38'])
analysis_conclusions = active_findings(conclusions)
display(findings_view(conclusions))


### 3.5 Выбор спецификации модели

**Входы**

- Результаты EDA, попарного анализа и проверки гипотез
- Постановка задачи и критерии оценивания

**Выходы**

- Варианты спецификации: состав признаков, обработка, семейства моделей и настройки для сравнения

**Что делаем**

Определим варианты устройства будущей модели на основе накопленных выводов

#### Какие варианты спецификации сравним

| Вариант | Основная идея |
| --- | --- |
| Dummy | Одинаковый прогноз для всех заявок — доля дефолтов в train |
| Логистическая регрессия | Складывает взвешенные вклады признаков и переводит результат в оценку риска |
| Случайный лес | Усредняет прогнозы множества деревьев решений |
| Градиентный бустинг на деревьях | Добавляет деревья постепенно: каждое уточняет общий результат |

Сравним линейную модель, лес и бустинг с базовым прогнозом Dummy

Оставляем все 11 признаков. Связь двух столбцов ещё не означает, что один из них бесполезен для прогноза. Грейд и ставку сохраняем: по нашей постановке они известны до прогноза

На этапе 5 обучим кандидатов на train, выберем модель и порог по правилам этапа 1.2. На этапе 6 оценим выбранное решение на test

## 4 Подготовка данных к обучению

**Входы**

- Данные после очистки и разбиения
- Спецификации моделей из этапа 3.5
- Таблица выводов и решения по признакам

**Выходы**

- Правила обработки и готовые входы для моделей

**Что делаем**

Заполним пропуски и переведём категории в понятный модели вид. Все нужные для этого значения считаем только по train

### 4.1 Состав признаков и выборки

Используем все 11 признаков из спецификаций этапа 3.5. Целевую переменную отделяем от входов модели

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler

NUMERICAL_FEATURES = [
    'person_age', 'person_income', 'person_emp_length', 'loan_amnt',
    'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
]
NOMINAL_FEATURES = ['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']
ORDINAL_FEATURES = ['loan_grade']
GRADE_ORDER = list('ABCDEFG')
FEATURES = NUMERICAL_FEATURES + NOMINAL_FEATURES + ORDINAL_FEATURES


In [ ]:
from sklearn.base import clone
from threadpoolctl import threadpool_limits

# Ограничиваем число вычислительных потоков для воспроизводимых запусков.
_ml_thread_limit = threadpool_limits(limits=1)

X_train = train.loc[:, FEATURES].copy()
y_train = train['loan_status'].astype(int).copy()
X_validation = validation.loc[:, FEATURES].copy()
y_validation = validation['loan_status'].astype(int).copy()
split_summary = pd.DataFrame([
    {'Выборка': name, 'Записей': len(frame), 'Доля дефолтов': frame['loan_status'].mean()}
    for name, frame in [('train', train), ('validation', validation), ('test', test)]
]).set_index('Выборка')
display(split_summary.round(4))


### 4.2 Числовые признаки

Пропуски заполняем медианами train и добавляем индикаторы пропусков. Затем вычитаем медиану и делим на межквартильный размах train — это масштабирование RobustScaler\
Задаём эти операции как цепочку Pipeline, которая обучается вместе с моделью

In [ ]:
def make_numerical_preprocessor():
    return Pipeline([
        ('impute', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
        ('scale', RobustScaler()),
    ])


In [ ]:
numerical_audit = make_numerical_preprocessor().fit(X_train[NUMERICAL_FEATURES])
display(pd.DataFrame({
    'Медиана для пропусков': numerical_audit.named_steps['impute'].statistics_,
}, index=NUMERICAL_FEATURES))


### 4.3 Номинальные признаки

Пропуски заполняем модой train. Каждую категорию кодируем отдельным столбцом 0/1 — one-hot-кодированием. Порядок категорий не задаём

In [ ]:
def make_nominal_preprocessor():
    return Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])


In [ ]:
nominal_audit = make_nominal_preprocessor().fit(X_train[NOMINAL_FEATURES])
display(pd.DataFrame({
    'Мода для пропусков': nominal_audit.named_steps['impute'].statistics_,
    'Категорий': [len(values) for values in nominal_audit.named_steps['encode'].categories_],
}, index=NOMINAL_FEATURES))


### 4.4 Порядковый признак

Для loan_grade сохраняем порядок A–G, кодируя его числами 0–6. Пропуски заполняем модой train\
Логистическая регрессия предполагает одинаковую прибавку к сумме вкладов признаков при переходе на одну категорию, деревья могут задавать разные границы разбиений

In [ ]:
def make_ordinal_preprocessor():
    return Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
        ('encode', OrdinalEncoder(categories=[GRADE_ORDER])),
    ])


In [ ]:
display(pd.DataFrame({'Грейд': GRADE_ORDER, 'Код': range(len(GRADE_ORDER))}))


### 4.5 Общая обработка

Объединяем три цепочки. Каждый кандидат при обучении получит собственную копию обработки, параметры которой определяются только по train\
Посмотрим результат обработки первых пяти строк

In [ ]:
def make_preprocessor():
    return ColumnTransformer([
        ('numerical', make_numerical_preprocessor(), NUMERICAL_FEATURES),
        ('nominal', make_nominal_preprocessor(), NOMINAL_FEATURES),
        ('ordinal', make_ordinal_preprocessor(), ORDINAL_FEATURES),
    ], remainder='drop', verbose_feature_names_out=True)


In [ ]:
preprocessor_template = make_preprocessor()
audit_preprocessor = clone(preprocessor_template).fit(X_train)
preprocessing_preview = pd.DataFrame(
    audit_preprocessor.transform(X_train.head(5)),
    columns=audit_preprocessor.get_feature_names_out(), index=X_train.head(5).index,
)
display(preprocessing_preview.round(3))


## 5 Обучение и выбор модели

**Входы**

- train, validation и правила обработки данных
- Спецификации моделей из этапа 3.5 и критерии оценивания из этапа 1.2

**Выходы**

- Обученная модель, выбранный порог и результаты сравнения

**Что делаем**

Обучим восемь вариантов на train. Выберем модель по AP на validation, затем подберём для неё порог. Test оставим для итоговой оценки

### 5.1 Кандидаты и настройки

Всего попробуем восемь вариантов. В подписях используем короткие имена: Logistic — логистическая регрессия, Forest — лес, Boosting — бустинг

| Модель | Варианты |
| --- | --- |
| Dummy | Один базовый прогноз |
| Логистическая регрессия | C=0.1, C=1 и C=1 с повышенным весом редкого класса |
| Лес | 200 деревьев; минимум 2 или 5 записей в конечной группе дерева, во втором случае — повышенный вес редкого класса |
| Бустинг | 200 шагов; до 15 или 31 конечной группы в каждом дереве |

**C** управляет ограничением коэффициентов логистической модели: меньше C — сильнее ограничение. В подписях **leaf** означает минимальное число записей в конечной группе дерева, **leaves** — максимальное число таких групп. **Balanced** означает, что ошибкам на редком классе при обучении придаётся больший вес

Все настройки задаём заранее. Для повторяемости фиксируем случайность числом 42. Каждый вариант обучаем вместе с обработкой данных на train

In [ ]:
from IPython.display import Markdown
'Метрики и порог: без доступа к test и без переобучения моделей'
import numpy as np
import pandas as pd
from sklearn.metrics import (

    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, f1_score, fbeta_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score,
)


def classification_metrics(y_true, probabilities, threshold=0.5):
    predictions = np.asarray(probabilities) >= threshold
    return {
        'AP': average_precision_score(y_true, probabilities),
        'ROC-AUC': roc_auc_score(y_true, probabilities),
        'Precision': precision_score(y_true, predictions, zero_division=0),
        'Recall': recall_score(y_true, predictions, zero_division=0),
        'F1': f1_score(y_true, predictions, zero_division=0),
        'F2': fbeta_score(y_true, predictions, beta=2, zero_division=0),
        'Accuracy': accuracy_score(y_true, predictions),
        'Balanced accuracy': balanced_accuracy_score(y_true, predictions),
        'Brier': brier_score_loss(y_true, probabilities),
    }


def select_f2_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    p, r = precision[:-1], recall[:-1]
    scores = np.divide(5 * p * r, 4 * p + r, out=np.zeros_like(p), where=(4 * p + r) > 0)
    # При равном F2 выбираем больший порог. Все входы — исключительно validation.
    positions = np.flatnonzero(np.isclose(scores, scores.max(), rtol=0, atol=1e-12))
    best = positions[-1]
    table = pd.DataFrame({'Порог': thresholds, 'Precision': p, 'Recall': r, 'F2': scores})
    return float(thresholds[best]), table


In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

model_candidates = {
    'Dummy: prior': DummyClassifier(strategy='prior'),
    'Logistic: C=0.1': LogisticRegression(C=.1, max_iter=2000, random_state=42),
    'Logistic: C=1': LogisticRegression(C=1, max_iter=2000, random_state=42),
    'Logistic: balanced': LogisticRegression(C=1, class_weight='balanced', max_iter=2000, random_state=42),
    'Forest: leaf=2': RandomForestClassifier(n_estimators=200, min_samples_leaf=2, max_features='sqrt', n_jobs=1, random_state=42),
    'Forest: leaf=5 balanced': RandomForestClassifier(n_estimators=200, min_samples_leaf=5, max_features='sqrt', class_weight='balanced', n_jobs=1, random_state=42),
    'Boosting: leaves=15': HistGradientBoostingClassifier(max_iter=200, max_leaf_nodes=15, learning_rate=.1, l2_regularization=1, early_stopping=False, random_state=42),
    'Boosting: leaves=31': HistGradientBoostingClassifier(max_iter=200, max_leaf_nodes=31, learning_rate=.1, l2_regularization=1, early_stopping=False, random_state=42),
}


### 5.2 Обучение

Каждый кандидат обучаем на train вместе с его обработкой данных. Затем получаем оценки риска на validation

In [ ]:
trained_models = {}
validation_probabilities = {}
comparison_records = []
for _name, _estimator in model_candidates.items():
    _pipeline = Pipeline([('preprocess', clone(preprocessor_template)), ('model', _estimator)])
    _pipeline.fit(X_train, y_train)
    _probability = _pipeline.predict_proba(X_validation)[:, 1]
    trained_models[_name] = _pipeline
    validation_probabilities[_name] = _probability
    comparison_records.append({'Модель': _name, **classification_metrics(y_validation, _probability)})


### 5.3 Выбор модели

Сравниваем кандидатов по AP на validation и выбираем вариант с наибольшим значением

In [ ]:
validation_comparison = pd.DataFrame(comparison_records).set_index('Модель').sort_values('AP', ascending=False, kind='stable')
selected_model_name = str(validation_comparison.index[0])
selected_pipeline = trained_models[selected_model_name]
selected_validation_probability = validation_probabilities[selected_model_name]
display(validation_comparison.round(4))


#### Выбранная модель

Выбираем **градиентный бустинг на деревьях — `Boosting: leaves=31`**: 200 шагов обучения, до 31 конечной группы в каждом дереве. Его AP на validation — **0,9016**, наибольшая среди восьми рассмотренных вариантов\
Для этого обученного экземпляра выберем порог, затем передадим модель и её обработку на итоговую оценку

In [ ]:
selected_estimator_parameters = selected_pipeline.named_steps['model'].get_params()
parameter_labels = {
    'strategy': 'Стратегия', 'C': 'C', 'class_weight': 'Вес классов',
    'n_estimators': 'Число деревьев', 'min_samples_leaf': 'Минимум записей в конечной группе',
    'max_features': 'Признаков на разбиение', 'max_iter': 'Максимум итераций',
    'max_leaf_nodes': 'Максимум конечных групп в дереве', 'learning_rate': 'Шаг обучения',
    'l2_regularization': 'Коэффициент L2', 'early_stopping': 'Досрочное завершение',
    'random_state': 'Начальное состояние генератора',
}
selected_parameters_table = pd.DataFrame([
    {'Настройка': label, 'Выбранное значение': str(selected_estimator_parameters[key])}
    for key, label in parameter_labels.items() if key in selected_estimator_parameters
]).set_index('Настройка')
display(selected_parameters_table)


### 5.4 Выбор порога

Для выбранной модели находим порог с максимальной F2 на validation. При равных значениях берём больший порог

In [ ]:
selected_threshold, threshold_curve = select_f2_threshold(y_validation, selected_validation_probability)
validation_threshold_comparison = pd.DataFrame([
    {'Правило': 'Порог 0.5', 'Порог': .5, **classification_metrics(y_validation, selected_validation_probability, .5)},
    {'Правило': 'Максимум F2 на validation', 'Порог': selected_threshold, **classification_metrics(y_validation, selected_validation_probability, selected_threshold)},
]).set_index('Правило')
display(validation_threshold_comparison.round(4))
# Зафиксированное решение. После этого fit больше не вызывается.
frozen_decision = {
    'model': selected_model_name, 'threshold': selected_threshold, 'selection_metric': 'validation AP',
    'fit_sample': 'train only', 'seed': 42, 'positive_class': 1,
}
display(Markdown(
    f'**Рабочий порог — {selected_threshold:.4f}**. '
    f'F2 на validation — **{validation_threshold_comparison.iloc[1]["F2"]:.4f}**. '
    'Модель, её обработку и порог фиксируем; на этапе 6 проверим это решение на test без изменений'
))


### 5.5 Результат сравнения

Сравним AP кандидатов и посмотрим, как F2, precision и recall выбранной модели зависят от порога

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
validation_comparison['AP'].sort_values().plot.barh(ax=axes[0], color='#4682a9')
axes[0].set(title='Выбор модели: только validation', xlabel='Average Precision', ylabel='')
axes[1].plot(threshold_curve['Порог'], threshold_curve['F2'], label='F2')
axes[1].plot(threshold_curve['Порог'], threshold_curve['Recall'], label='Полнота (Recall)', alpha=.7)
axes[1].plot(threshold_curve['Порог'], threshold_curve['Precision'], label='Точность (Precision)', alpha=.7)
axes[1].axvline(selected_threshold, color='black', linestyle='--', label='Выбранный порог')
axes[1].set(title='Выбор порога: только validation', xlabel='Порог', ylabel='Метрика', ylim=(0, 1.02))
axes[1].legend()
plt.tight_layout()
plt.show()


## 6 Итоговая оценка

**Входы**

- Выбранная модель, порог и test
- Метрики и правила оценивания из постановки задачи

**Выходы**

- Значения заданных метрик, графики и разбор ошибок

**Что делаем**

Рассчитаем на test метрики, заданные в этапе 1.2, и разберём ошибки. Модель и порог на этом шаге уже не меняем

### 6.1 Метрики и ошибки

Сравним выбранную модель с Dummy на одном test по метрикам этапа 1.2. В матрице ошибок строки — фактический исход, столбцы — прогноз

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve
from sklearn.calibration import calibration_curve

X_test = test.loc[:, FEATURES].copy()
y_test = test['loan_status'].astype(int).copy()
test_probability = selected_pipeline.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= selected_threshold).astype(int)
dummy_test_probability = trained_models['Dummy: prior'].predict_proba(X_test)[:, 1]
test_metrics = pd.DataFrame([
    {'Решение': 'Dummy, порог 0.5', **classification_metrics(y_test, dummy_test_probability)},
    {'Решение': f'{selected_model_name}, порог validation', **classification_metrics(y_test, test_probability, selected_threshold)},
]).set_index('Решение')
test_confusion = confusion_matrix(y_test, test_prediction, labels=[0, 1])
tn, fp, fn, tp = test_confusion.ravel().tolist()
display(test_metrics.round(4))
display(pd.DataFrame(test_confusion, index=['Факт: без дефолта', 'Факт: дефолт'], columns=['Прогноз: без дефолта', 'Прогноз: дефолт']))


### 6.2 Разброс оценок

Используем bootstrap — выбор строк с повторами: 300 раз пересобираем test, сохраняя численность каждого класса. Показываем центральные 95% полученных значений метрик\
Модель и порог фиксированы. Эти интервалы не учитывают новое обучение или изменение данных со временем

In [ ]:
# Стратифицированный bootstrap уже готовых прогнозов: модель/порог не меняются.
_bootstrap_rng = np.random.default_rng(42)
_bootstrap_y = y_test.to_numpy()
_negative_indices = np.flatnonzero(_bootstrap_y == 0)
_positive_indices = np.flatnonzero(_bootstrap_y == 1)
_bootstrap_metrics = []
for _ in range(300):
    _indices = np.concatenate([
        _bootstrap_rng.choice(_negative_indices, len(_negative_indices), replace=True),
        _bootstrap_rng.choice(_positive_indices, len(_positive_indices), replace=True),
    ])
    _bootstrap_metrics.append(classification_metrics(_bootstrap_y[_indices], test_probability[_indices], selected_threshold))
bootstrap_metrics = pd.DataFrame(_bootstrap_metrics)
metric_intervals = pd.DataFrame({
    'Оценка': test_metrics.iloc[1],
    'Нижняя 95%': bootstrap_metrics.quantile(.025),
    'Верхняя 95%': bootstrap_metrics.quantile(.975),
})
display(metric_intervals.round(4))


### 6.3 Диагностика прогнозов

ROC-кривая сопоставляет recall с долей ложных тревог $FP/(FP+TN)$, PR-кривая — precision с recall при разных порогах

График калибровки показывает, насколько оценки риска согласуются с фактическими частотами дефолта

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
_fpr, _tpr, _ = roc_curve(y_test, test_probability)
axes[0, 0].plot(_fpr, _tpr, label=f'ROC-AUC = {test_metrics.iloc[1]["ROC-AUC"]:.3f}')
axes[0, 0].plot([0, 1], [0, 1], '--', color='grey')
axes[0, 0].set(title='ROC: test', xlabel='Доля ложных тревог', ylabel='Полнота')
axes[0, 0].legend()
_precision, _recall, _ = precision_recall_curve(y_test, test_probability)
axes[0, 1].plot(_recall, _precision, label=f'AP = {test_metrics.iloc[1]["AP"]:.3f}')
axes[0, 1].axhline(y_test.mean(), linestyle='--', color='grey', label='Доля дефолтов test')
axes[0, 1].set(title='Precision–Recall: test', xlabel='Полнота (Recall)', ylabel='Точность (Precision)')
axes[0, 1].legend()
ConfusionMatrixDisplay(test_confusion, display_labels=['Без дефолта', 'Дефолт']).plot(ax=axes[1, 0], colorbar=False, cmap='Blues')
axes[1, 0].set(title='Матрица ошибок: выбранный порог', xlabel='Прогноз', ylabel='Факт')
_observed, _predicted = calibration_curve(y_test, test_probability, n_bins=8, strategy='quantile')
axes[1, 1].plot(_predicted, _observed, 'o-', label='Модель')
axes[1, 1].plot([0, 1], [0, 1], '--', color='grey', label='Идеальная калибровка')
axes[1, 1].set(title='Диагностика калибровки: test', xlabel='Средний прогноз', ylabel='Наблюдаемая доля дефолтов')
axes[1, 1].legend()
plt.tight_layout()
plt.show()


In [ ]:
display(Markdown(
    f'На test ({len(test):,} записей) AP = **{test_metrics.iloc[1]["AP"]:.4f}**, '
    f'ROC-AUC = **{test_metrics.iloc[1]["ROC-AUC"]:.4f}**. '
    f'При зафиксированном пороге обнаружено **{tp} из {tp + fn}** дефолтов; '
    f'пропущено **{fn}**, ложных тревог — **{fp}** из {tn + fp} кредитов без дефолта. '
    'После просмотра test решение не перенастраиваем'
))


#### Что ограничивает выводы

Результат получен на одном случайном разбиении Kaggle-датасета.\
Неизвестны время заявок, период наблюдения дефолта и повторы клиентов. Сходство с данными конкретного банка не проверяли; редкие грейды представлены небольшим числом записей

Грейд и ставка должны быть известны до прогноза. Если нужна оценка до их назначения, состав признаков придётся пересмотреть и провести новый эксперимент

Перед практическим применением проверим модель на более поздних и внешних данных, оценим стоимость ошибок и качество для разных групп заёмщиков

## 7 Прогноз для новых заявок и итоговые выводы

**Входы**

- Обученная модель и новые заявки без известного исхода

**Выходы**

- Оценки риска и классы заявок
- Итоговые выводы и условия дальнейшего применения

**Что делаем**

Получим прогноз для трёх придуманных заявок

#### Как получить прогноз

Передаём таблицу с теми же 11 признаками, но без `loan_status`. Для каждой строки получаем оценку риска и класс по выбранному порогу\
Применяем ту же обработку, что при обучении: заполняем пропуски, кодируем категории и масштабируем числовые признаки

In [ ]:
def prepare_applications(applications):
    'Проверка новых заявок; строки не удаляем, целевую переменную не принимаем'
    if not applications.columns.is_unique or set(applications.columns) != set(FEATURES):
        raise ValueError('Неверный набор столбцов: нужны ровно 11 признаков без loan_status')
    result = applications.loc[:, FEATURES].copy()
    for column in NUMERICAL_FEATURES:
        result[column] = pd.to_numeric(result[column]).astype(float)
    # То же правило очистки стажа, что до разбиения; на инференсе не теряем строки.
    employment = result['person_emp_length']
    result.loc[(employment < 0) | (employment > 86), 'person_emp_length'] = np.nan
    invalid = (
        (result['person_age'] < 18) | (result['person_age'] > 100)
        | (result['person_income'] <= 0) | (result['loan_amnt'] <= 0)
        | (result['loan_int_rate'] <= 0)
        | (result['loan_percent_income'] < 0) | (result['loan_percent_income'] > 1)
        | (result['cb_person_cred_hist_length'] < 0)
    )
    if invalid.any():
        raise ValueError('Недопустимые числовые значения: исправьте заявку до прогноза')
    for column in NOMINAL_FEATURES + ORDINAL_FEATURES:
        result[column] = result[column].astype(object).where(result[column].notna(), np.nan)
    return result


In [ ]:
def predict_applications(applications, pipeline, threshold):
    prepared = prepare_applications(applications)
    scores = pipeline.predict_proba(prepared)[:, 1]
    return pd.DataFrame({
        'Оценка риска дефолта': scores,
        'Класс при выбранном пороге': (scores >= threshold).astype(int),
    }, index=applications.index)


new_applications = pd.DataFrame([
    [28, 65000, 4, 9000, 10.8, .14, 6, 'RENT', 'EDUCATION', 'N', 'B'],
    [42, 38000, 8, 16000, 17.5, .42, 12, 'MORTGAGE', 'DEBTCONSOLIDATION', 'Y', 'E'],
    [31, 92000, np.nan, 5000, np.nan, .05, 8, 'OWN', 'PERSONAL', 'N', 'A'],
], columns=FEATURES, index=['Заявка 1', 'Заявка 2', 'Заявка 3'])
example_predictions = predict_applications(new_applications, selected_pipeline, selected_threshold)
display(new_applications)
display(example_predictions.round(4))


#### За чем следить при дальнейшем использовании

| Что проверять | Зачем |
| --- | --- |
| Поля, единицы измерения, пропуски и новые категории | Чтобы входные данные не меняли смысл |
| Изменение распределений по сравнению с train | Чтобы заметить, что новые заявки стали другими |
| Качество после появления фактических исходов | Чтобы понять, сохраняются ли точность, полнота и калибровка |
| Стоимость ошибок и качество для разных групп | Чтобы оценить последствия применения модели |
| Версии данных, обработки и порога | Чтобы знать, какой именно вариант дал прогноз |

Покажем проверку пропусков на трёх заявках, сравнив их с train. Для оценки изменений распределений понадобится поток реальных заявок

In [ ]:
monitoring_reference = pd.DataFrame({
    'Доля пропусков train': X_train.isna().mean(),
    'Доля пропусков примера': new_applications.isna().mean(),
})
display(monitoring_reference.round(4))


### Итоговая таблица выводов

In [ ]:
_final_updates = [
    ('preprocessing', 'Оставили все 11 признаков. Числовые пропуски заполнили медианами train и отметили отдельными столбцами, категориальные — самым частым значением; грейды закодировали по порядку A–G' + '. В исходных данных пропуски стажа — 2,75%, ставки — 9,56%; масштабы числовых признаков различаются', 'Применять ту же цепочку обработки к новым заявкам. Редкие высокие доходы не удалять'),
    ('model_selection', f"На validation из {len(validation_comparison)} заранее заданных кандидатов выбран {selected_model_name}; AP = {validation_comparison.iloc[0]['AP']:.4f}", 'Сравнили только эти варианты на одном разбиении. Другие модели или настройки могут оказаться лучше'),
    ('threshold', f"Порог {selected_threshold:.4f} выбран по F2 только на validation; на test Recall = {test_metrics.iloc[1]['Recall']:.4f}, Precision = {test_metrics.iloc[1]['Precision']:.4f}", 'При выборе порога отдаём приоритет поиску дефолтов. Для практики нужно учесть стоимость ошибок и проверить порог на новых данных'),
    ('test_result', f"Test: AP = {test_metrics.iloc[1]['AP']:.4f}, ROC-AUC = {test_metrics.iloc[1]['ROC-AUC']:.4f}, F2 = {test_metrics.iloc[1]['F2']:.4f}; пропусков дефолта {fn}, ложных тревог {fp}", 'После этой оценки модель не меняем. Разброс метрик показан в этапе 6'),
    ('uncertainty', 'Bootstrap показывает разброс метрик для готовой модели на этом test. Новое обучение и данные другого банка он не учитывает', 'Не знаем время заявок и повторы клиентов; проверки на внешних данных нет. Работа в других условиях ещё не проверена'),
    ('deployment', 'Можно передать сразу несколько заявок и получить для каждой риск и класс. Неверный набор полей вызывает ошибку; новая номинальная категория даёт нули в своих столбцах one-hot', 'Перед применением нужно проверить доступность полей, калибровку и качество на новых данных и в разных группах. Прогноз сам по себе не является решением об отказе'),
]
_final_updates.extend([
    ('f01', f'Исходно {len(data_raw)} записей, 11 признаков и loan_status. После удаления повторов и недопустимого возраста — {len(data)} записей',
     'Сохраняем правила очистки и разбиение 70/15/15'),
    ('f33', conclusions.loc['f33', 'Выводы'],
     'Сравнили более гибкие модели на validation; выбрали бустинг. По этому сравнению нельзя выделить отдельный вклад суммы или дохода и установить причину связи'),
])
conclusions = update_findings(conclusions, _final_updates, remove=['f03', 'f06', 'f26'])
# Для итоговой таблицы оставляем только актуальные версии без подсветки изменений
conclusions = update_findings(conclusions)
final_conclusions = active_findings(conclusions)
display(findings_view(conclusions))
